# Proyecto 1 - Obtención y Limpieza de Datos

## Universidad del Valle de Guatemala

### CC3084 - Data Science

---

## Objetivo

Automatizar la obtención de los establecimientos educativos del Ministerio de Educación de Guatemala correspondientes al nivel **Diversificado** para todos los departamentos del país.

Todo el proceso debe ser reproducible mediante código.

---

Autor: Osman Emanuel de León García


In [8]:
# Instalación de librerías

%pip install selenium webdriver-manager pandas openpyxl xlrd

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
import time
import re
from pathlib import Path

import numpy as np
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.common.exceptions import TimeoutException
from selenium.common.exceptions import NoSuchElementException

from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

In [10]:
# -----------------------------
# Configuración del proyecto
# -----------------------------

from pathlib import Path

URL = "https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/wbfBuscar.aspx"

# Estructura de carpetas
DATA_FOLDER = Path("data")

RAW_FOLDER = DATA_FOLDER / "raw"
CSV_FOLDER = DATA_FOLDER / "csv"
CLEAN_FOLDER = DATA_FOLDER / "clean"

RAW_FOLDER.mkdir(parents=True, exist_ok=True)
CSV_FOLDER.mkdir(parents=True, exist_ok=True)
CLEAN_FOLDER.mkdir(parents=True, exist_ok=True)

DOWNLOAD_TIMEOUT = 60

In [11]:
departamentos = {
    "16": "Alta_Verapaz",
    "15": "Baja_Verapaz",
    "04": "Chimaltenango",
    "20": "Chiquimula",
    "00": "Ciudad_Capital",
    "02": "El_Progreso",
    "05": "Escuintla",
    "01": "Guatemala",
    "13": "Huehuetenango",
    "18": "Izabal",
    "21": "Jalapa",
    "22": "Jutiapa",
    "17": "Peten",
    "09": "Quetzaltenango",
    "14": "Quiche",
    "11": "Retalhuleu",
    "03": "Sacatepequez",
    "12": "San_Marcos",
    "06": "Santa_Rosa",
    "07": "Solola",
    "10": "Suchitepequez",
    "08": "Totonicapan",
    "19": "Zacapa"
}

In [12]:
def esperar_elemento(driver, by, value, timeout=20):
    """
    Espera hasta que un elemento esté presente.
    """
    return WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((by, value))
    )


def esperar_click(driver, by, value, timeout=20):
    """
    Espera hasta que un elemento sea clickeable.
    """
    return WebDriverWait(driver, timeout).until(
        EC.element_to_be_clickable((by, value))
    )


def esperar_descarga(carpeta, timeout=60):

    tiempo = time.time()

    while time.time() - tiempo < timeout:

        archivos_tmp = list(Path(carpeta).glob("*.crdownload"))

        if not archivos_tmp:

            archivo = Path(carpeta) / "establecimiento.xls"

            if archivo.exists():
                return archivo

        time.sleep(1)

    raise TimeoutException("La descarga tardó demasiado.")

In [13]:
def renombrar_excel(nombre_departamento):
    """
    Renombra el archivo descargado con el nombre del departamento.
    """

    archivo = RAW_FOLDER / "establecimiento.xls"

    nuevo = RAW_FOLDER / f"{nombre_departamento}.xls"

    if nuevo.exists():
        nuevo.unlink()

    archivo.rename(nuevo)

    return nuevo

In [14]:
options = webdriver.ChromeOptions()

prefs = {
    "download.default_directory": str(RAW_FOLDER.resolve()),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}

options.add_experimental_option("prefs", prefs)

options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

wait = WebDriverWait(driver,20)

driver.get(URL)

In [15]:
for codigo, nombre in departamentos.items():

    print("="*60)
    print(f"Descargando {nombre}")
    print("="*60)

    # Departamento

    Select(
        esperar_elemento(
            driver,
            By.ID,
            "_ctl0_ContentPlaceHolder1_cmbDepartamento"
        )
    ).select_by_value(codigo)

    # Nivel

    Select(
        esperar_elemento(
            driver,
            By.ID,
            "_ctl0_ContentPlaceHolder1_cmbNivel"
        )
    ).select_by_value("46")

    # Buscar

    esperar_click(
        driver,
        By.ID,
        "_ctl0_ContentPlaceHolder1_IbtnConsultar"
    ).click()

    # Esperar que aparezca el botón Exportar

    esperar_click(
        driver,
        By.ID,
        "_ctl0_ContentPlaceHolder1_btnExportar",
        timeout=30
    )

    # Exportar

    driver.find_element(
        By.ID,
        "_ctl0_ContentPlaceHolder1_btnExportar"
    ).click()

    # Esperar descarga

    esperar_descarga(RAW_FOLDER, DOWNLOAD_TIMEOUT)

    # Renombrar

    renombrar_excel(nombre)

    print(f"{nombre} descargado correctamente.\n")

Descargando Alta_Verapaz
Alta_Verapaz descargado correctamente.

Descargando Baja_Verapaz
Baja_Verapaz descargado correctamente.

Descargando Chimaltenango
Chimaltenango descargado correctamente.

Descargando Chiquimula
Chiquimula descargado correctamente.

Descargando Ciudad_Capital
Ciudad_Capital descargado correctamente.

Descargando El_Progreso
El_Progreso descargado correctamente.

Descargando Escuintla
Escuintla descargado correctamente.

Descargando Guatemala
Guatemala descargado correctamente.

Descargando Huehuetenango
Huehuetenango descargado correctamente.

Descargando Izabal
Izabal descargado correctamente.

Descargando Jalapa
Jalapa descargado correctamente.

Descargando Jutiapa
Jutiapa descargado correctamente.

Descargando Peten
Peten descargado correctamente.

Descargando Quetzaltenango
Quetzaltenango descargado correctamente.

Descargando Quiche
Quiche descargado correctamente.

Descargando Retalhuleu
Retalhuleu descargado correctamente.

Descargando Sacatepequez
Sacat

In [16]:
print("="*50)
print("Descarga completada.")
print(f"Departamentos descargados: {len(departamentos)}")
print(f"Archivos guardados en: {RAW_FOLDER.resolve()}")
print("="*50)

Descarga completada.
Departamentos descargados: 23
Archivos guardados en: C:\Users\Emadlg\Desktop\UVG\Cuarto Año\Segundo Ciclo\Data Science\Data_Science\Proyecto1\data\raw


# Parte 2 - Conversión y Unión de los Datos

En esta sección se leerán todos los archivos Excel descargados automáticamente,
se convertirán a DataFrames de pandas y posteriormente se unirán en un único
conjunto de datos que servirá como base para el proceso de limpieza.

In [17]:
%pip install lxml html5lib beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from glob import glob

In [19]:
# Buscar todos los Excel descargados

excel_files = sorted(glob(str(RAW_FOLDER / "*.xls")))

print(f"Se encontraron {len(excel_files)} archivos.")

Se encontraron 23 archivos.


In [20]:
dataframes = []

for archivo in excel_files:

    print(f"Leyendo {Path(archivo).name}")

    tablas = pd.read_html(archivo)

    tabla_correcta = None

    for tabla in tablas:

        # Buscar si la PRIMERA FILA contiene "ESTABLECIMIENTO"
        primera_fila = tabla.iloc[0].astype(str).str.upper().tolist()

        if "ESTABLECIMIENTO" in primera_fila:

            tabla_correcta = tabla.copy()
            break

    if tabla_correcta is None:
        print(f"No se encontró la tabla de datos en {archivo}")
        continue

    # La primera fila realmente son los encabezados
    tabla_correcta.columns = tabla_correcta.iloc[0]

    # Eliminar esa fila del contenido
    tabla_correcta = tabla_correcta.iloc[1:].reset_index(drop=True)

    tabla_correcta["ARCHIVO_ORIGEN"] = Path(archivo).stem

    dataframes.append(tabla_correcta)

Leyendo Alta_Verapaz.xls
Leyendo Baja_Verapaz.xls
Leyendo Chimaltenango.xls
Leyendo Chiquimula.xls
Leyendo Ciudad_Capital.xls
Leyendo El_Progreso.xls
Leyendo Escuintla.xls
Leyendo Guatemala.xls
Leyendo Huehuetenango.xls
Leyendo Izabal.xls
Leyendo Jalapa.xls
Leyendo Jutiapa.xls
Leyendo Peten.xls
Leyendo Quetzaltenango.xls
Leyendo Quiche.xls
Leyendo Retalhuleu.xls
Leyendo Sacatepequez.xls
Leyendo San_Marcos.xls
Leyendo Santa_Rosa.xls
Leyendo Solola.xls
Leyendo Suchitepequez.xls
Leyendo Totonicapan.xls
Leyendo Zacapa.xls


In [21]:
len(dataframes)

23

In [22]:
df_raw = pd.concat(dataframes, ignore_index=True)

df_raw.head()

,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL,ARCHIVO_ORIGEN
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,NaN,JORGE EDUARDO PAQUE LAZARO,--,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ,Alta_Verapaz
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,Alta_Verapaz
2,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,58015763,ARMANDO AUDILIO POP CUCUL,GILMA DOLORES GUAY PAZ DE LEAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,Alta_Verapaz
3,16-01-0140-46,16-01-0926,ALTA VERAPAZ,COBAN,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,78232301,JOSE ARTURO CHOC CHEN,VIRGINIA SOLANO SERRANO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,Alta_Verapaz
4,16-01-0141-46,16-01-0924,ALTA VERAPAZ,COBAN,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,40389968,MOISÉS ADRIÁN LÓPEZ PÉREZ,ESTELA DEL CARMEN QUIM ROSALES,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,Alta_Verapaz


In [23]:
print(f"Departamentos cargados: {len(dataframes)}")

Departamentos cargados: 23


In [24]:
df_raw = pd.concat(
    dataframes,
    ignore_index=True
)

In [25]:
print(df_raw.shape)

(11891, 18)


In [26]:
csv_path = CSV_FOLDER / "establecimientos_crudos.csv"

df_raw.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("CSV generado correctamente.")
print(csv_path)

CSV generado correctamente.
data\csv\establecimientos_crudos.csv


In [27]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 11891 entries, 0 to 11890
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CODIGO           11868 non-null  str  
 1   DISTRITO         11336 non-null  str  
 2   DEPARTAMENTO     11868 non-null  str  
 3   MUNICIPIO        11868 non-null  str  
 4   ESTABLECIMIENTO  11863 non-null  str  
 5   DIRECCION        11792 non-null  str  
 6   TELEFONO         10922 non-null  str  
 7   SUPERVISOR       11333 non-null  str  
 8   DIRECTOR         10137 non-null  str  
 9   NIVEL            11868 non-null  str  
 10  SECTOR           11868 non-null  str  
 11  AREA             11868 non-null  str  
 12  STATUS           11868 non-null  str  
 13  MODALIDAD        11868 non-null  str  
 14  JORNADA          11868 non-null  str  
 15  PLAN             11868 non-null  str  
 16  DEPARTAMENTAL    11868 non-null  str  
 17  ARCHIVO_ORIGEN   11891 non-null  str  
dtypes: str(18)
memory

In [28]:
print("="*50)

print(f"Registros : {df_raw.shape[0]}")
print(f"Variables : {df_raw.shape[1]}")

print("="*50)

Registros : 11891
Variables : 18


# Parte 3 - Diagnóstico del Conjunto de Datos

En esta sección se realiza un análisis exploratorio del conjunto de datos obtenido, con el objetivo de identificar problemas de calidad antes de iniciar el proceso de limpieza.

Se evaluarán:

- Número de registros y variables.
- Tipos de datos.
- Valores faltantes.
- Valores únicos.
- Registros duplicados.
- Posibles inconsistencias.

In [29]:
print("="*60)
print("DIMENSIONES DEL DATASET")
print("="*60)

print(f"Registros : {df_raw.shape[0]}")
print(f"Variables : {df_raw.shape[1]}")

DIMENSIONES DEL DATASET
Registros : 11891
Variables : 18


In [30]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 11891 entries, 0 to 11890
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CODIGO           11868 non-null  str  
 1   DISTRITO         11336 non-null  str  
 2   DEPARTAMENTO     11868 non-null  str  
 3   MUNICIPIO        11868 non-null  str  
 4   ESTABLECIMIENTO  11863 non-null  str  
 5   DIRECCION        11792 non-null  str  
 6   TELEFONO         10922 non-null  str  
 7   SUPERVISOR       11333 non-null  str  
 8   DIRECTOR         10137 non-null  str  
 9   NIVEL            11868 non-null  str  
 10  SECTOR           11868 non-null  str  
 11  AREA             11868 non-null  str  
 12  STATUS           11868 non-null  str  
 13  MODALIDAD        11868 non-null  str  
 14  JORNADA          11868 non-null  str  
 15  PLAN             11868 non-null  str  
 16  DEPARTAMENTAL    11868 non-null  str  
 17  ARCHIVO_ORIGEN   11891 non-null  str  
dtypes: str(18)
memory

In [31]:
tipos = pd.DataFrame({
    "Variable": df_raw.columns,
    "Tipo de dato": df_raw.dtypes.astype(str)
})

tipos

,Variable,Tipo de dato
0,,
CODIGO,CODIGO,str
DISTRITO,DISTRITO,str
DEPARTAMENTO,DEPARTAMENTO,str
MUNICIPIO,MUNICIPIO,str
ESTABLECIMIENTO,ESTABLECIMIENTO,str
DIRECCION,DIRECCION,str
TELEFONO,TELEFONO,str
SUPERVISOR,SUPERVISOR,str
DIRECTOR,DIRECTOR,str


In [32]:
faltantes = pd.DataFrame({
    "Variable": df_raw.columns,
    "Valores faltantes": df_raw.isna().sum(),
    "Porcentaje": (
        df_raw.isna().sum() / len(df_raw) * 100
    ).round(2)
})

faltantes.sort_values(
    "Valores faltantes",
    ascending=False
)

,Variable,Valores faltantes,Porcentaje
0,,,
DIRECTOR,DIRECTOR,1754,14.75
TELEFONO,TELEFONO,969,8.15
SUPERVISOR,SUPERVISOR,558,4.69
DISTRITO,DISTRITO,555,4.67
DIRECCION,DIRECCION,99,0.83
ESTABLECIMIENTO,ESTABLECIMIENTO,28,0.24
CODIGO,CODIGO,23,0.19
MUNICIPIO,MUNICIPIO,23,0.19
DEPARTAMENTO,DEPARTAMENTO,23,0.19


In [33]:
unicos = pd.DataFrame({
    "Variable": df_raw.columns,
    "Valores únicos": df_raw.nunique()
})

unicos

,Variable,Valores únicos
0,,
CODIGO,CODIGO,11868
DISTRITO,DISTRITO,1681
DEPARTAMENTO,DEPARTAMENTO,23
MUNICIPIO,MUNICIPIO,352
ESTABLECIMIENTO,ESTABLECIMIENTO,6307
DIRECCION,DIRECCION,7437
TELEFONO,TELEFONO,6574
SUPERVISOR,SUPERVISOR,1283
DIRECTOR,DIRECTOR,5520


In [34]:
duplicados = df_raw.duplicated().sum()

print(f"Duplicados exactos: {duplicados}")

Duplicados exactos: 0


In [35]:
texto = df_raw.select_dtypes(include="object").columns

espacios = {}

for columna in texto:

    espacios[columna] = (
        df_raw[columna]
        .fillna("")
        .astype(str)
        .str.startswith(" ")
        .sum()
        +
        df_raw[columna]
        .fillna("")
        .astype(str)
        .str.endswith(" ")
        .sum()
    )

pd.DataFrame.from_dict(
    espacios,
    orient="index",
    columns=["Registros con espacios"]
)

C:\Users\Emadlg\AppData\Local\Temp\ipykernel_29600\3430575785.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  texto = df_raw.select_dtypes(include="object").columns


,Registros con espacios
CODIGO,0
DISTRITO,0
DEPARTAMENTO,0
MUNICIPIO,0
ESTABLECIMIENTO,0
DIRECCION,0
TELEFONO,0
SUPERVISOR,0
DIRECTOR,0
NIVEL,0


In [36]:
vacias = {}

for columna in texto:

    vacias[columna] = (
        df_raw[columna]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

pd.DataFrame.from_dict(
    vacias,
    orient="index",
    columns=["Cadenas vacías"]
)

,Cadenas vacías
CODIGO,23
DISTRITO,555
DEPARTAMENTO,23
MUNICIPIO,23
ESTABLECIMIENTO,28
DIRECCION,99
TELEFONO,969
SUPERVISOR,558
DIRECTOR,1754
NIVEL,23


In [37]:
resumen = pd.DataFrame({

    "Métrica":[

        "Registros",
        "Variables",
        "Duplicados exactos",
        "Variables con NA",
        "Valores faltantes"

    ],

    "Valor":[

        len(df_raw),
        df_raw.shape[1],
        df_raw.duplicated().sum(),
        (df_raw.isna().sum()>0).sum(),
        df_raw.isna().sum().sum()

    ]

})

resumen

,Métrica,Valor
0,Registros,11891
1,Variables,18
2,Duplicados exactos,0
3,Variables con NA,17
4,Valores faltantes,4216


# Parte 4 - Plan de Limpieza

Antes de modificar el conjunto de datos, se identifican los principales problemas de calidad detectados durante el diagnóstico y se establece una estrategia de limpieza para cada variable.

El objetivo es garantizar que el proceso sea transparente, reproducible y que preserve la integridad de la información original.

In [38]:
print("="*60)
print("DESCRIPCIÓN DEL CONJUNTO DE DATOS")
print("="*60)

print(f"Registros : {df_raw.shape[0]}")
print(f"Variables : {df_raw.shape[1]}")

DESCRIPCIÓN DEL CONJUNTO DE DATOS
Registros : 11891
Variables : 18


El conjunto de datos contiene la información de los establecimientos educativos autorizados por el Ministerio de Educación de Guatemala que imparten el nivel Diversificado. Los datos provienen de los 23 departamentos del país y fueron obtenidos automáticamente mediante Selenium desde el portal oficial del MINEDUC.

In [39]:
df_clean = df_raw.copy()

In [40]:
variables_limpieza = pd.DataFrame({
    "Variable": [
        "ESTABLECIMIENTO",
        "DIRECCION",
        "TELEFONO",
        "DIRECTOR",
        "SUPERVISOR",
        "DEPARTAMENTO",
        "MUNICIPIO",
        "MODALIDAD",
        "JORNADA",
        "STATUS"
    ],

    "Problemas identificados": [
        "Espacios adicionales, posibles duplicados parciales y diferencias de escritura.",
        "Espacios múltiples, abreviaturas y formatos inconsistentes.",
        "Valores faltantes, diferentes formatos y posibles caracteres inválidos.",
        "Espacios innecesarios y diferencias de escritura.",
        "Espacios innecesarios y diferencias de escritura.",
        "Posibles diferencias de escritura y espacios.",
        "Posibles diferencias de escritura y espacios.",
        "Posibles categorías inconsistentes.",
        "Posibles categorías inconsistentes.",
        "Posibles categorías inconsistentes."
    ]
})

variables_limpieza

,Variable,Problemas identificados
0,ESTABLECIMIENTO,"Espacios adicionales, posibles duplicados parc..."
1,DIRECCION,"Espacios múltiples, abreviaturas y formatos in..."
2,TELEFONO,"Valores faltantes, diferentes formatos y posib..."
3,DIRECTOR,Espacios innecesarios y diferencias de escritura.
4,SUPERVISOR,Espacios innecesarios y diferencias de escritura.
5,DEPARTAMENTO,Posibles diferencias de escritura y espacios.
6,MUNICIPIO,Posibles diferencias de escritura y espacios.
7,MODALIDAD,Posibles categorías inconsistentes.
8,JORNADA,Posibles categorías inconsistentes.
9,STATUS,Posibles categorías inconsistentes.


In [41]:
plan_limpieza = pd.DataFrame({

    "Variable":[
        "CODIGO",
        "DISTRITO",
        "DEPARTAMENTO",
        "MUNICIPIO",
        "ESTABLECIMIENTO",
        "DIRECCION",
        "TELEFONO",
        "SUPERVISOR",
        "DIRECTOR",
        "NIVEL",
        "SECTOR",
        "AREA",
        "STATUS",
        "MODALIDAD",
        "JORNADA",
        "PLAN",
        "DEPARTAMENTAL"
    ],

    "Problema detectado":[

        "Espacios innecesarios.",

        "Espacios innecesarios.",

        "Espacios y posibles diferencias de escritura.",

        "Espacios y posibles diferencias de escritura.",

        "Espacios múltiples y posibles duplicados parciales.",

        "Espacios múltiples y formatos inconsistentes.",

        "Valores faltantes y formatos inconsistentes.",

        "Espacios adicionales.",

        "Espacios adicionales.",

        "Verificar que todos correspondan al nivel Diversificado.",

        "Posibles categorías inconsistentes.",

        "Posibles categorías inconsistentes.",

        "Posibles categorías inconsistentes.",

        "Posibles categorías inconsistentes.",

        "Posibles categorías inconsistentes.",

        "Posibles categorías inconsistentes.",

        "Verificar consistencia con la variable DEPARTAMENTO."
    ],

    "Transformación propuesta":[

        "Eliminar espacios al inicio y final.",

        "Eliminar espacios al inicio y final.",

        "Eliminar espacios y unificar formato del texto.",

        "Eliminar espacios y unificar formato del texto.",

        "Eliminar espacios múltiples, normalizar texto y revisar posibles duplicados parciales.",

        "Eliminar espacios múltiples y normalizar formato.",

        "Convertir a tipo texto, normalizar formato e identificar valores faltantes.",

        "Eliminar espacios innecesarios.",

        "Eliminar espacios innecesarios.",

        "Validar que todos los registros pertenezcan al nivel Diversificado.",

        "Revisar y unificar categorías.",

        "Revisar y unificar categorías.",

        "Revisar y unificar categorías.",

        "Revisar y unificar categorías.",

        "Revisar y unificar categorías.",

        "Revisar y unificar categorías.",

        "Comparar con DEPARTAMENTO para verificar consistencia."
    ],

    "Riesgo":[

        "Bajo",

        "Bajo",

        "Bajo",

        "Bajo",

        "Medio",

        "Bajo",

        "Medio",

        "Bajo",

        "Bajo",

        "Ninguno",

        "Bajo",

        "Bajo",

        "Bajo",

        "Bajo",

        "Bajo",

        "Bajo",

        "Bajo"
    ]

})

plan_limpieza

,Variable,Problema detectado,Transformación propuesta,Riesgo
0,CODIGO,Espacios innecesarios.,Eliminar espacios al inicio y final.,Bajo
1,DISTRITO,Espacios innecesarios.,Eliminar espacios al inicio y final.,Bajo
2,DEPARTAMENTO,Espacios y posibles diferencias de escritura.,Eliminar espacios y unificar formato del texto.,Bajo
3,MUNICIPIO,Espacios y posibles diferencias de escritura.,Eliminar espacios y unificar formato del texto.,Bajo
4,ESTABLECIMIENTO,Espacios múltiples y posibles duplicados parci...,"Eliminar espacios múltiples, normalizar texto ...",Medio
5,DIRECCION,Espacios múltiples y formatos inconsistentes.,Eliminar espacios múltiples y normalizar formato.,Bajo
6,TELEFONO,Valores faltantes y formatos inconsistentes.,"Convertir a tipo texto, normalizar formato e i...",Medio
7,SUPERVISOR,Espacios adicionales.,Eliminar espacios innecesarios.,Bajo
8,DIRECTOR,Espacios adicionales.,Eliminar espacios innecesarios.,Bajo
9,NIVEL,Verificar que todos correspondan al nivel Dive...,Validar que todos los registros pertenezcan al...,Ninguno


# Parte 5 - Limpieza del Conjunto de Datos

En esta sección se aplican las transformaciones definidas en el plan de limpieza.

Todas las modificaciones realizadas serán registradas para garantizar que el proceso sea reproducible y transparente.

In [42]:
# Copia del dataset crudo
df_clean = df_raw.copy()

In [43]:
registro_transformaciones = []

def registrar(variable, problema, transformacion, afectados, justificacion):

    registro_transformaciones.append({
        "Variable": variable,
        "Problema": problema,
        "Transformación": transformacion,
        "Registros afectados": afectados,
        "Justificación": justificacion
    })

In [44]:
columnas_originales = df_clean.columns.tolist()

df_clean.columns = (
    df_clean.columns
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(" ", "_")
)

registrar(
    "TODAS LAS COLUMNAS",
    "Nombres inconsistentes",
    "Normalización de nombres",
    len(df_clean.columns),
    "Facilitar el acceso mediante pandas."
)

In [45]:
print("="*70)
print("LIMPIEZA DE ESPACIOS")
print("="*70)

columnas_texto = df_clean.select_dtypes(
    include=["object", "string"]
).columns.tolist()

for columna in columnas_texto:

    antes = df_clean[columna].copy()

    df_clean[columna] = (
        df_clean[columna]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    afectados = (antes != df_clean[columna]).fillna(False).sum()

    print(f"{columna:<20} {afectados:>6} registros modificados")

    registrar(
        columna,
        "Espacios iniciales, finales o múltiples",
        "Eliminación de espacios innecesarios",
        int(afectados),
        "Uniformar el formato sin alterar el contenido."
    )

LIMPIEZA DE ESPACIOS
CODIGO                    0 registros modificados
DISTRITO                  0 registros modificados
DEPARTAMENTO              0 registros modificados
MUNICIPIO                 0 registros modificados
ESTABLECIMIENTO           0 registros modificados
DIRECCION                 0 registros modificados
TELEFONO                  0 registros modificados
SUPERVISOR                0 registros modificados
DIRECTOR                  0 registros modificados
NIVEL                     0 registros modificados
SECTOR                    0 registros modificados
AREA                      0 registros modificados
STATUS                    0 registros modificados
MODALIDAD                 0 registros modificados
JORNADA                   0 registros modificados
PLAN                      0 registros modificados
DEPARTAMENTAL             0 registros modificados
ARCHIVO_ORIGEN            0 registros modificados


In [46]:
print("="*70)
print("CONVERSIÓN DE VALORES VACÍOS")
print("="*70)

valores_vacios = [
    "",
    " ",
    "N/A",
    "NA",
    "NULL",
    "-",
    "--",
    ".",
    "nan"
]

for columna in columnas_texto:

    antes = df_clean[columna].isna().sum()

    df_clean[columna] = df_clean[columna].replace(
        valores_vacios,
        np.nan
    )

    despues = df_clean[columna].isna().sum()

    afectados = despues - antes

    print(f"{columna:<20} {afectados:>6} valores convertidos")

    registrar(
        columna,
        "Valores vacíos",
        "Conversión a NaN",
        int(afectados),
        "Representar correctamente los datos faltantes."
    )

CONVERSIÓN DE VALORES VACÍOS
CODIGO                    0 valores convertidos
DISTRITO                  0 valores convertidos
DEPARTAMENTO              0 valores convertidos
MUNICIPIO                 0 valores convertidos
ESTABLECIMIENTO           0 valores convertidos
DIRECCION                 6 valores convertidos
TELEFONO                  0 valores convertidos
SUPERVISOR                0 valores convertidos
DIRECTOR                 72 valores convertidos
NIVEL                     0 valores convertidos
SECTOR                    0 valores convertidos
AREA                      0 valores convertidos
STATUS                    0 valores convertidos
MODALIDAD                 0 valores convertidos
JORNADA                   0 valores convertidos
PLAN                      0 valores convertidos
DEPARTAMENTAL             0 valores convertidos
ARCHIVO_ORIGEN            0 valores convertidos


In [47]:
print("="*70)
print("TIPO DE DATO")
print("="*70)

if "TELEFONO" in df_clean.columns:

    df_clean["TELEFONO"] = (
        df_clean["TELEFONO"]
        .astype("string")
    )

    print("TELEFONO convertido a string.")

    registrar(
        "TELEFONO",
        "Tipo incorrecto",
        "Conversión a string",
        len(df_clean),
        "Los teléfonos no deben tratarse como números."
    )

TIPO DE DATO
TELEFONO convertido a string.


In [48]:
print("="*70)
print("DUPLICADOS")
print("="*70)

duplicados = df_clean.duplicated()

print(f"Duplicados exactos encontrados: {duplicados.sum()}")

registrar(
    "DATASET",
    "Duplicados exactos",
    "Identificación",
    int(duplicados.sum()),
    "Se identifican para su análisis posterior."
)

DUPLICADOS
Duplicados exactos encontrados: 0


In [49]:
for columna in [

    "DEPARTAMENTO",
    "MUNICIPIO",
    "SECTOR",
    "AREA",
    "STATUS",
    "MODALIDAD",
    "JORNADA"

]:

    print("="*60)

    print(columna)

    print(df_clean[columna].value_counts())

DEPARTAMENTO
DEPARTAMENTO
CIUDAD CAPITAL    2161
GUATEMALA         1908
SAN MARCOS         724
ESCUINTLA          708
HUEHUETENANGO      591
QUETZALTENANGO     551
PETEN              516
ALTA VERAPAZ       475
SUCHITEPEQUEZ      437
CHIMALTENANGO      435
SACATEPEQUEZ       430
IZABAL             413
JUTIAPA            392
RETALHULEU         364
QUICHE             323
CHIQUIMULA         239
SANTA ROSA         213
SOLOLA             192
JALAPA             183
BAJA VERAPAZ       171
EL PROGRESO        158
ZACAPA             156
TOTONICAPAN        128
Name: count, dtype: Int64
MUNICIPIO
MUNICIPIO
ZONA 1                         868
MIXCO                          562
VILLA NUEVA                    446
QUETZALTENANGO                 281
ZONA 7                         236
                              ... 
SAN JOSE CHACAYA                 1
SAN ANTONIO PALOPO               1
SAN MARCOS LA LAGUNA             1
SAN MIGUEL PANAM                 1
SAN BARTOLO AGUAS CALIENTES      1
Name: count, L

In [50]:
registro = pd.DataFrame(registro_transformaciones)

registro.index = registro.index + 1

registro

,Variable,Problema,Transformación,Registros afectados,Justificación
1,TODAS LAS COLUMNAS,Nombres inconsistentes,Normalización de nombres,18,Facilitar el acceso mediante pandas.
2,CODIGO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
3,DISTRITO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
4,DEPARTAMENTO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
5,MUNICIPIO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
6,ESTABLECIMIENTO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
7,DIRECCION,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
8,TELEFONO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
9,SUPERVISOR,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
10,DIRECTOR,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.


In [51]:
registro.to_csv(

    CLEAN_FOLDER / "registro_transformaciones.csv",

    index=False,

    encoding="utf-8-sig"

)

print("Registro de transformaciones guardado.")

Registro de transformaciones guardado.


## Auditoría de categorías

Antes de modificar variables categóricas se inspeccionan todos sus valores únicos para identificar diferencias de escritura, espacios, categorías duplicadas o valores inesperados.

In [52]:
categoricas = [

    "DEPARTAMENTO",
    "MUNICIPIO",
    "SECTOR",
    "AREA",
    "STATUS",
    "MODALIDAD",
    "JORNADA",
    "PLAN",
    "NIVEL"

]

In [53]:
for columna in categoricas:

    print("="*80)
    print(columna)
    print("="*80)

    print(df_clean[columna].value_counts(dropna=False))

    print("\n")

DEPARTAMENTO
DEPARTAMENTO
CIUDAD CAPITAL    2161
GUATEMALA         1908
SAN MARCOS         724
ESCUINTLA          708
HUEHUETENANGO      591
QUETZALTENANGO     551
PETEN              516
ALTA VERAPAZ       475
SUCHITEPEQUEZ      437
CHIMALTENANGO      435
SACATEPEQUEZ       430
IZABAL             413
JUTIAPA            392
RETALHULEU         364
QUICHE             323
CHIQUIMULA         239
SANTA ROSA         213
SOLOLA             192
JALAPA             183
BAJA VERAPAZ       171
EL PROGRESO        158
ZACAPA             156
TOTONICAPAN        128
<NA>                23
Name: count, dtype: Int64


MUNICIPIO
MUNICIPIO
ZONA 1                         868
MIXCO                          562
VILLA NUEVA                    446
QUETZALTENANGO                 281
ZONA 7                         236
                              ... 
SAN JOSE CHACAYA                 1
SAN ANTONIO PALOPO               1
SAN MARCOS LA LAGUNA             1
SAN MIGUEL PANAM                 1
SAN BARTOLO AGUAS CALIEN

In [54]:
print(sorted(df_clean["DEPARTAMENTO"].dropna().unique()))

['ALTA VERAPAZ', 'BAJA VERAPAZ', 'CHIMALTENANGO', 'CHIQUIMULA', 'CIUDAD CAPITAL', 'EL PROGRESO', 'ESCUINTLA', 'GUATEMALA', 'HUEHUETENANGO', 'IZABAL', 'JALAPA', 'JUTIAPA', 'PETEN', 'QUETZALTENANGO', 'QUICHE', 'RETALHULEU', 'SACATEPEQUEZ', 'SAN MARCOS', 'SANTA ROSA', 'SOLOLA', 'SUCHITEPEQUEZ', 'TOTONICAPAN', 'ZACAPA']


In [55]:
municipios = sorted(df_clean["MUNICIPIO"].dropna().unique())

print(f"Municipios encontrados: {len(municipios)}")

municipios

Municipios encontrados: 352


['ACATENANGO',
 'AGUA BLANCA',
 'AGUACATAN',
 'ALMOLONGA',
 'ALOTENANGO',
 'AMATITLAN',
 'ANTIGUA GUATEMALA',
 'ASUNCION MITA',
 'ATESCATEMPA',
 'AYUTLA',
 'BARBERENA',
 'CABAÑAS',
 'CABRICAN',
 'CAJOLA',
 'CAMOTAN',
 'CANILLA',
 'CANTEL',
 'CASILLAS',
 'CATARINA',
 'CHAHAL',
 'CHAJUL',
 'CHAMPERICO',
 'CHIANTLA',
 'CHICACAO',
 'CHICAMAN',
 'CHICHE',
 'CHIMALTENANGO',
 'CHINAUTLA',
 'CHINIQUE',
 'CHIQUIMULA',
 'CHIQUIMULILLA',
 'CHISEC',
 'CHUARRANCHO',
 'CIUDAD VIEJA',
 'COATEPEQUE',
 'COBAN',
 'COLOMBA COSTA CUCA',
 'COLOTENANGO',
 'COMAPA',
 'COMITANCILLO',
 'CONCEPCION CHIQUIRICHAPA',
 'CONCEPCION HUISTA',
 'CONCEPCION LAS MINAS',
 'CONCEPCION TUTUAPA',
 'CONGUACO',
 'CUBULCO',
 'CUILAPA',
 'CUILCO',
 'CUNEN',
 'CUYOTENANGO',
 'DOLORES',
 'EL ADELANTO',
 'EL ASINTAL',
 'EL CHAL',
 'EL ESTOR',
 'EL JICARO',
 'EL PALMAR',
 'EL PROGRESO',
 'EL QUETZAL',
 'EL TEJAR',
 'EL TUMBADOR',
 'ESCUINTLA',
 'ESQUIPULAS',
 'ESQUIPULAS PALO GORDO',
 'ESTANZUELA',
 'FLORES',
 'FLORES COSTA CUCA',
 

In [56]:
df_clean["MODALIDAD"].value_counts(dropna=False)

MODALIDAD
MONOLINGUE    11394
BILINGUE        474
<NA>             23
Name: count, dtype: Int64

In [57]:
df_clean["JORNADA"].value_counts(dropna=False)

JORNADA
DOBLE          3772
VESPERTINA     3407
MATUTINA       3045
SIN JORNADA    1100
NOCTURNA        415
INTERMEDIA      129
<NA>             23
Name: count, dtype: Int64

In [58]:
df_clean["SECTOR"].value_counts(dropna=False)

SECTOR
PRIVADO        9892
OFICIAL        1499
COOPERATIVA     298
MUNICIPAL       179
<NA>             23
Name: count, dtype: Int64

In [59]:
df_clean["AREA"].value_counts(dropna=False)

AREA
URBANA             9461
RURAL              2404
<NA>                 23
SIN ESPECIFICAR       3
Name: count, dtype: Int64

In [60]:
df_clean["STATUS"].value_counts(dropna=False)

STATUS
ABIERTA                    6861
CERRADA TEMPORALMENTE      3005
CERRADA DEFINITIVAMENTE    1842
TEMPORAL TITULOS            110
TEMPORAL NOMBRAMIENTO        50
<NA>                         23
Name: count, dtype: Int64

In [61]:
df_clean["NIVEL"].value_counts(dropna=False)

NIVEL
DIVERSIFICADO    11868
<NA>                23
Name: count, dtype: Int64

In [62]:
df_clean["DIRECCION"].sample(20, random_state=42)

10157                                  CABECERA MUNICIPAL
3295                     21 CALLE 36-40 COLONIA SAN JORGE
2701                      3A. CALLE 2-28 COLONIA LANDIVAR
3312               27 CALLE Y 34 AVENIDA, COLONIA KENNEDY
7525                               4TA. CALLE 2-11 ZONA 3
9385            LOTIFICACION EL INFIERNITO CABALLO BLANCO
385                                      BARRIO EL CENTRO
9296                               ALDEA NUEVA CANDELARIA
1626                                      8A. CALLE 12-46
4863                4A AVE. B 0-69, COLONIA COTIO, ZONA 2
6372                               3A AVENIDA 1-40 ZONA 3
3107                                 13 AVENIDA "A" 26-64
7642                                    BARRIO LA FEDERAL
3315     LOTE 101 Y 102 SECCIÓN "C" COLONIA LAS ILUSIONES
9763                   3A. CALLE COLONIA LOS ANGELES NO.8
5517                                        5A. AVE. 0-84
856                               1A. AVENIDA 1-80 ZONA 3
1166          

In [63]:
def auditar_columna(df, columna):

    print("="*80)
    print(columna)
    print("="*80)

    print(f"Valores únicos : {df[columna].nunique()}")

    print(f"Nulos          : {df[columna].isna().sum()}")

    print("\nFrecuencia:\n")

    display(df[columna].value_counts(dropna=False))

    print("\n")

In [64]:
for columna in categoricas:
    auditar_columna(df_clean, columna)

DEPARTAMENTO
Valores únicos : 23
Nulos          : 23

Frecuencia:



DEPARTAMENTO
CIUDAD CAPITAL    2161
GUATEMALA         1908
SAN MARCOS         724
ESCUINTLA          708
HUEHUETENANGO      591
QUETZALTENANGO     551
PETEN              516
ALTA VERAPAZ       475
SUCHITEPEQUEZ      437
CHIMALTENANGO      435
SACATEPEQUEZ       430
IZABAL             413
JUTIAPA            392
RETALHULEU         364
QUICHE             323
CHIQUIMULA         239
SANTA ROSA         213
SOLOLA             192
JALAPA             183
BAJA VERAPAZ       171
EL PROGRESO        158
ZACAPA             156
TOTONICAPAN        128
<NA>                23
Name: count, dtype: Int64



MUNICIPIO
Valores únicos : 352
Nulos          : 23

Frecuencia:



MUNICIPIO
ZONA 1                         868
MIXCO                          562
VILLA NUEVA                    446
QUETZALTENANGO                 281
ZONA 7                         236
                              ... 
SAN JOSE CHACAYA                 1
SAN ANTONIO PALOPO               1
SAN MARCOS LA LAGUNA             1
SAN MIGUEL PANAM                 1
SAN BARTOLO AGUAS CALIENTES      1
Name: count, Length: 353, dtype: Int64



SECTOR
Valores únicos : 4
Nulos          : 23

Frecuencia:



...

JORNADA
DOBLE          3772
VESPERTINA     3407
MATUTINA       3045
SIN JORNADA    1100
NOCTURNA        415
INTERMEDIA      129
<NA>             23
Name: count, dtype: Int64



PLAN
Valores únicos : 13
Nulos          : 23

Frecuencia:



PLAN
DIARIO(REGULAR)                          7452
FIN DE SEMANA                            2898
SEMIPRESENCIAL (FIN DE SEMANA)            542
SEMIPRESENCIAL (UN DÍA A LA SEMANA)       438
A DISTANCIA                               188
SEMIPRESENCIAL                            118
VIRTUAL A DISTANCIA                        70
SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)      67
SABATINO                                   59
DOMINICAL                                  27
<NA>                                       23
MIXTO                                       4
IRREGULAR                                   3
INTERCALADO                                 2
Name: count, dtype: Int64



NIVEL
Valores únicos : 1
Nulos          : 23

Frecuencia:



NIVEL
DIVERSIFICADO    11868
<NA>                23
Name: count, dtype: Int64

## Parte 5  - Limpieza avanzada

Se completan los aspectos de limpieza que la guía exige y que aún no se habían cubierto:
filas vacías, normalización de mayúsculas, formato de teléfonos, validación de dominio
geográfico contra el catálogo oficial de Guatemala, duplicados parciales y consistencia
entre variables.

In [65]:
# Instalar librerías necesarias para normalización y comparación de cadenas
%pip install rapidfuzz unidecode -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [66]:
from unidecode import unidecode
from rapidfuzz import fuzz

### 5.1 Filas completamente vacías

In [67]:
print("="*70)
print("5.1 FILAS COMPLETAMENTE VACÍAS")
print("="*70)

cols_contenido = [c for c in df_clean.columns if c != "ARCHIVO_ORIGEN"]
filas_vacias = df_clean[cols_contenido].isna().all(axis=1)
n_vacias = int(filas_vacias.sum())

print(f"Filas totalmente vacías detectadas: {n_vacias}")
print("Corresponden a una fila de resumen/pie que el sitio del MINEDUC agrega")
print("al final de la tabla exportada de cada departamento (23 departamentos = 23 filas).")

df_clean = df_clean[~filas_vacias].reset_index(drop=True)

registrar(
    "TODAS LAS COLUMNAS",
    "Filas de resumen vacías generadas por la exportación del portal",
    "Eliminación de filas sin contenido real",
    n_vacias,
    "Cada archivo Excel exportado por el MINEDUC agrega una fila de pie sin datos; "
    "no corresponde a ningún establecimiento y debe excluirse del análisis."
)

print(f"Filas restantes: {len(df_clean)}")

5.1 FILAS COMPLETAMENTE VACÍAS
Filas totalmente vacías detectadas: 23
Corresponden a una fila de resumen/pie que el sitio del MINEDUC agrega
al final de la tabla exportada de cada departamento (23 departamentos = 23 filas).
Filas restantes: 11868


### 5.2 Normalización de texto: mayúsculas uniformes

In [68]:
print("="*70)
print("5.2 NORMALIZACIÓN DE TEXTO (MAYÚSCULAS)")
print("="*70)

columnas_texto_norm = [
    c for c in df_clean.select_dtypes(include=["string"]).columns
    if c != "ARCHIVO_ORIGEN"
]

for columna in columnas_texto_norm:

    antes = df_clean[columna].copy()

    df_clean[columna] = df_clean[columna].str.upper()

    afectados = (antes != df_clean[columna]).fillna(False).sum()

    if afectados > 0:
        print(f"{columna:<20} {afectados:>6} registros convertidos a mayúsculas")

        registrar(
            columna,
            "Uso inconsistente de mayúsculas/minúsculas",
            "Conversión a mayúsculas",
            int(afectados),
            "Unificar el formato de presentación del texto sin alterar su contenido."
        )

5.2 NORMALIZACIÓN DE TEXTO (MAYÚSCULAS)
DIRECCION                10 registros convertidos a mayúsculas
TELEFONO                  1 registros convertidos a mayúsculas


### 5.3 Formato de teléfonos

`TELEFONO` trae formatos muy heterogéneos: números únicos de 8 dígitos, varios números
separados por coma/guion/slash/"Y", texto como "FAX" o "EXT.", y longitudes distintas a 8
dígitos. Se crea una versión normalizada que extrae únicamente los números de 8 dígitos
válidos, conservando el original para trazabilidad.

In [69]:
print("="*70)
print("5.3 FORMATO DE TELÉFONOS")
print("="*70)

def extraer_telefonos(valor):
    """Extrae los números de teléfono de 8 dígitos válidos contenidos en el texto."""
    if pd.isna(valor):
        return pd.NA, False, 0

    grupos = re.findall(r"\d{7,8}", str(valor))
    validos = [g for g in grupos if len(g) == 8]

    if not validos:
        return pd.NA, False, 0

    return "; ".join(validos), True, len(validos)

resultado = df_clean["TELEFONO"].apply(extraer_telefonos)

df_clean["TELEFONO_NORMALIZADO"] = [r[0] for r in resultado]
df_clean["TELEFONO_VALIDO"] = [r[1] for r in resultado]

n_multiples = sum(1 for r in resultado if r[2] > 1)
telefono_no_vacio = df_clean["TELEFONO"].notna()
n_invalidos = int((telefono_no_vacio & ~df_clean["TELEFONO_VALIDO"].fillna(False)).sum())

print(f"Teléfonos con al menos un número válido de 8 dígitos : {df_clean['TELEFONO_VALIDO'].sum()}")
print(f"Teléfonos con más de un número (líneas múltiples/fax): {n_multiples}")
print(f"Teléfonos no vacíos sin ningún número de 8 dígitos    : {n_invalidos}")

registrar(
    "TELEFONO",
    "Formatos heterogéneos: números múltiples separados por coma/guion/slash, texto (FAX, EXT), longitudes distintas a 8 dígitos",
    "Se crea TELEFONO_NORMALIZADO (dígitos válidos de 8 cifras, separados por '; ') y TELEFONO_VALIDO (booleano)",
    int(df_clean["TELEFONO"].notna().sum()),
    "Preservar el dato original y exponer una versión utilizable para análisis, sin descartar información."
)

5.3 FORMATO DE TELÉFONOS
Teléfonos con al menos un número válido de 8 dígitos : 10817
Teléfonos con más de un número (líneas múltiples/fax): 122
Teléfonos no vacíos sin ningún número de 8 dígitos    : 105


# Reporte de Formatos Inconsistentes

En esta sección se verifica que las variables de texto tengan un formato consistente después del proceso de limpieza.

In [89]:
def reporte_formatos(df):

    columnas = df.select_dtypes(include=["object", "string"]).columns

    reporte = []

    for columna in columnas:

        espacios_inicio = (
            df[columna]
            .fillna("")
            .astype(str)
            .str.startswith(" ")
            .sum()
        )

        espacios_final = (
            df[columna]
            .fillna("")
            .astype(str)
            .str.endswith(" ")
            .sum()
        )

        espacios_dobles = (
            df[columna]
            .fillna("")
            .astype(str)
            .str.contains(r"\s{2,}", regex=True)
            .sum()
        )

        reporte.append({

            "Variable": columna,
            "Espacios al inicio": espacios_inicio,
            "Espacios al final": espacios_final,
            "Espacios múltiples": espacios_dobles

        })

    return pd.DataFrame(reporte)

In [90]:
formatos = reporte_formatos(df_clean)

formatos

,Variable,Espacios al inicio,Espacios al final,Espacios múltiples
0,CODIGO,0,0,0
1,DISTRITO,0,0,0
2,DEPARTAMENTO,0,0,0
3,MUNICIPIO,0,0,0
4,ESTABLECIMIENTO,0,0,0
5,DIRECCION,0,0,0
6,TELEFONO,0,0,0
7,SUPERVISOR,0,0,0
8,DIRECTOR,0,0,0
9,NIVEL,0,0,0


In [91]:
formatos.to_csv(

    CLEAN_FOLDER / "reporte_formatos.csv",

    index=False,

    encoding="utf-8-sig"

)

### 5.4 Validación de dominio: DEPARTAMENTO y MUNICIPIO

Se construye un catálogo oficial con los 22 departamentos y 340 municipios de Guatemala
(fuente: INE / recopilación pública) para verificar que **todos** los valores de
`DEPARTAMENTO` y `MUNICIPIO` correspondan a lugares reales, y que cada municipio
pertenezca al departamento correcto.

In [70]:
# Catálogo oficial: 22 departamentos y 340 municipios de Guatemala
# Fuente: Instituto Nacional de Estadística (INE) / recopilación pública.
#
# Nota importante: el portal del MINEDUC trata "CIUDAD CAPITAL" como si fuera un
# departamento aparte de "GUATEMALA" para fines de búsqueda administrativa.
# Oficialmente "Ciudad Capital" ES el municipio de Guatemala (dentro del
# departamento de Guatemala). Se documenta esta particularidad y se añade como
# categoría administrativa válida adicional, ya que así opera la fuente de datos.

CATALOGO_MUNICIPIOS = {
    "ALTA VERAPAZ": [
        "COBAN", "SANTA CRUZ VERAPAZ", "SAN CRISTOBAL VERAPAZ", "TACTIC",
        "TAMAHU", "SAN MIGUEL TUCURU", "PANZOS", "SENAHU", "SAN PEDRO CARCHA",
        "SAN JUAN CHAMELCO", "SAN AGUSTIN LANQUIN", "SANTA MARIA CAHABON",
        "CHISEC", "CHAHAL", "FRAY BARTOLOME DE LAS CASAS",
        "SANTA CATALINA LA TINTA", "RAXRUHA",
    ],
    "BAJA VERAPAZ": [
        "SALAMA", "SAN MIGUEL CHICAJ", "RABINAL", "CUBULCO", "GRANADOS",
        "SANTA CRUZ EL CHOL", "SAN JERONIMO", "PURULHA",
    ],
    "CHIMALTENANGO": [
        "CHIMALTENANGO", "SAN JOSE POAQUIL", "SAN MARTIN JILOTEPEQUE",
        "SAN JUAN COMALAPA", "SANTA APOLONIA", "TECPAN GUATEMALA", "TECPAN",
        "PATZUN", "SAN MIGUEL POCHUTA", "PATZICIA", "SANTA CRUZ BALANYA",
        "ACATENANGO", "SAN PEDRO YEPOCAPA", "SAN ANDRES ITZAPA", "PARRAMOS",
        "ZARAGOZA", "EL TEJAR",
    ],
    "CHIQUIMULA": [
        "CHIQUIMULA", "JOCOTAN", "ESQUIPULAS", "CAMOTAN", "QUEZALTEPEQUE",
        "OLOPA", "IPALA", "SAN JUAN ERMITA", "CONCEPCION LAS MINAS",
        "SAN JACINTO", "SAN JOSE LA ARADA",
    ],
    "EL PROGRESO": [
        "EL JICARO", "MORAZAN", "SAN AGUSTIN ACASAGUASTLAN",
        "SAN ANTONIO LA PAZ", "SAN CRISTOBAL ACASAGUASTLAN", "SANARATE",
        "GUASTATOYA", "SANSARE",
    ],
    "ESCUINTLA": [
        "ESCUINTLA", "SANTA LUCIA COTZUMALGUAPA", "LA DEMOCRACIA",
        "SIQUINALA", "MASAGUA", "TIQUISATE", "LA GOMERA", "GUANAGAZAPA",
        "SAN JOSE", "IZTAPA", "PALIN", "SAN VICENTE PACAYA",
        "NUEVA CONCEPCION",
    ],
    "GUATEMALA": [
        "GUATEMALA", "SANTA CATARINA PINULA", "SAN JOSE PINULA",
        "SAN JOSE DEL GOLFO", "PALENCIA", "CHINAUTLA", "SAN PEDRO AYAMPUC",
        "MIXCO", "SAN PEDRO SACATEPEQUEZ", "SAN JUAN SACATEPEQUEZ",
        "CHUARRANCHO", "SAN RAYMUNDO", "FRAIJANES", "AMATITLAN",
        "VILLA NUEVA", "VILLA CANALES", "SAN MIGUEL PETAPA",
        "CIUDAD CAPITAL",
    ],
    "HUEHUETENANGO": [
        "HUEHUETENANGO", "CHIANTLA", "MALACATANCITO", "CUILCO", "NENTON",
        "SAN PEDRO NECTA", "JACALTENANGO", "SAN PEDRO SOLOMA",
        "SAN ILDEFONSO IXTAHUACAN", "SANTA BARBARA", "LA LIBERTAD",
        "LA DEMOCRACIA", "SAN MIGUEL ACATAN", "SAN RAFAEL LA INDEPENDENCIA",
        "TODOS SANTOS CUCHUMATAN", "SAN JUAN ATITAN", "SANTA EULALIA",
        "SAN MATEO IXTATAN", "COLOTENANGO", "SAN SEBASTIAN HUEHUETENANGO",
        "TECTITAN", "CONCEPCION HUISTA", "SAN JUAN IXCOY",
        "SAN ANTONIO HUISTA", "SANTA CRUZ BARILLAS", "SAN SEBASTIAN COATAN",
        "AGUACATAN", "SAN RAFAEL PETZAL", "SAN GASPAR IXCHIL",
        "SANTIAGO CHIMALTENANGO", "SANTA ANA HUISTA", "PETATAN",
        "UNION CANTINIL",
    ],
    "IZABAL": [
        "MORALES", "LOS AMATES", "LIVINGSTON", "EL ESTOR", "PUERTO BARRIOS",
    ],
    "JALAPA": [
        "JALAPA", "SAN PEDRO PINULA", "SAN LUIS JILOTEPEQUE",
        "SAN MANUEL CHAPARRON", "SAN CARLOS ALZATATE", "MONJAS",
        "MATAQUESCUINTLA",
    ],
    "JUTIAPA": [
        "JUTIAPA", "EL PROGRESO", "SANTA CATARINA MITA", "AGUA BLANCA",
        "ASUNCION MITA", "YUPILTEPEQUE", "ATESCATEMPA", "JEREZ",
        "EL ADELANTO", "ZAPOTITLAN", "COMAPA", "JALPATAGUA", "CONGUACO",
        "MOYUTA", "PASACO", "QUESADA",
    ],
    "PETEN": [
        "FLORES", "SAN JOSE", "SAN BENITO", "SAN ANDRES", "LA LIBERTAD",
        "SAN FRANCISCO", "SANTA ANA", "DOLORES", "SAN LUIS", "SAYAXCHE",
        "MELCHOR DE MENCOS", "POPTUN", "EL CHAL", "LAS CRUCES",
    ],
    "QUETZALTENANGO": [
        "QUETZALTENANGO", "SALCAJA", "SAN JUAN OLINTEPEQUE", "SAN CARLOS SIJA",
        "SIBILIA", "CABRICAN", "CAJOLA", "SAN MIGUEL SIGUILA",
        "SAN JUAN OSTUNCALCO", "SAN MATEO", "CONCEPCION CHIQUIRICHAPA",
        "SAN MARTIN SACATEPEQUEZ", "ALMOLONGA", "CANTEL", "HUITAN", "ZUNIL",
        "COLOMBA", "COLOMBA COSTA CUCA", "SAN FRANCISCO LA UNION", "EL PALMAR",
        "COATEPEQUE", "GENOVA", "FLORES COSTA CUCA", "LA ESPERANZA",
        "PALESTINA DE LOS ALTOS",
    ],
    "QUICHE": [
        "SANTA CRUZ DEL QUICHE", "CHICHE", "CHINIQUE", "ZACUALPA", "CHAJUL",
        "CHICHICASTENANGO", "SANTO TOMAS CHICHICASTENANGO", "PATZITE",
        "SAN ANTONIO ILOTENANGO", "SAN PEDRO JOCOPILAS", "CUNEN",
        "SAN JUAN COTZAL", "JOYABAJ", "SANTA MARIA JOYABAJ", "NEBAJ",
        "SANTA MARIA NEBAJ", "SAN ANDRES SAJCABAJA", "USPANTAN", "SACAPULAS",
        "SAN BARTOLOME JOCOTENANGO", "CANILLA", "CHICAMAN", "IXCAN",
        "PACHALUM",
    ],
    "RETALHULEU": [
        "RETALHULEU", "SAN SEBASTIAN", "SANTA CRUZ MULUA",
        "SAN MARTIN ZAPOTITLAN", "SAN FELIPE", "SAN ANDRES VILLA SECA",
        "CHAMPERICO", "NUEVO SAN CARLOS", "EL ASINTAL",
    ],
    "SACATEPEQUEZ": [
        "ANTIGUA GUATEMALA", "JOCOTENANGO", "PASTORES", "SUMPANGO",
        "SANTO DOMINGO XENACOJ", "SANTIAGO SACATEPEQUEZ",
        "SAN BARTOLOME MILPAS ALTAS", "SAN LUCAS SACATEPEQUEZ",
        "SANTA LUCIA MILPAS ALTAS", "MAGDALENA MILPAS ALTAS",
        "SANTA MARIA DE JESUS", "CIUDAD VIEJA", "SAN MIGUEL DUEÑAS",
        "SAN JUAN ALOTENANGO", "ALOTENANGO", "SAN ANTONIO AGUAS CALIENTES",
        "SANTA CATARINA BARAHONA",
    ],
    "SAN MARCOS": [
        "SAN MARCOS", "AYUTLA", "CATARINA", "COMITANCILLO",
        "CONCEPCION TUTUAPA", "EL QUETZAL", "EL RODEO", "SAN JOSE EL RODEO",
        "EL TUMBADOR", "IXCHIGUAN", "LA REFORMA", "MALACATAN",
        "NUEVO PROGRESO", "OCOS", "PAJAPITA", "ESQUIPULAS PALO GORDO",
        "SAN ANTONIO SACATEPEQUEZ", "SAN CRISTOBAL CUCHO",
        "SAN JOSE OJETENAM", "SAN LORENZO", "SAN MIGUEL IXTAHUACAN",
        "SAN PABLO", "SAN PEDRO SACATEPEQUEZ", "SAN RAFAEL PIE DE LA CUESTA",
        "SIBINAL", "SIPACAPA", "TACANA", "TAJUMULCO", "TEJUTLA",
        "RIO BLANCO", "LA BLANCA",
    ],
    "SANTA ROSA": [
        "CUILAPA", "BARBERENA", "CASILLAS", "CHIQUIMULILLA", "GUAZACAPAN",
        "NUEVA SANTA ROSA", "ORATORIO", "PUEBLO NUEVO VIÑAS",
        "SAN JUAN TECUACO", "SAN RAFAEL LAS FLORES", "SANTA CRUZ NARANJO",
        "SANTA MARIA IXHUATAN", "SANTA ROSA DE LIMA", "TAXISCO",
    ],
    "SOLOLA": [
        "SOLOLA", "CONCEPCION", "NAHUALA", "PANAJACHEL",
        "SAN ANDRES SEMETABAJ", "SAN ANTONIO PALOPO", "SAN JOSE CHACAYA",
        "SAN JUAN LA LAGUNA", "SAN LUCAS TOLIMAN", "SAN MARCOS LA LAGUNA",
        "SAN PABLO LA LAGUNA", "SAN PEDRO LA LAGUNA",
        "SANTA CATARINA IXTAHUACAN", "SANTA CATARINA PALOPO",
        "SANTA CLARA LA LAGUNA", "SANTA CRUZ LA LAGUNA",
        "SANTA LUCIA UTATLAN", "SANTA MARIA VISITACION", "SANTIAGO ATITLAN",
    ],
    "SUCHITEPEQUEZ": [
        "MAZATENANGO", "CUYOTENANGO", "SAN FRANCISCO ZAPOTITLAN",
        "SAN BERNARDINO", "SAN JOSE EL IDOLO", "SANTO DOMINGO SUCHITEPEQUEZ",
        "SAN LORENZO", "SAMAYAC", "SAN PABLO JOCOPILAS",
        "SAN ANTONIO SUCHITEPEQUEZ", "SAN MIGUEL PANAN", "SAN GABRIEL",
        "CHICACAO", "PATULUL", "SANTA BARBARA", "SAN JUAN BAUTISTA",
        "SANTO TOMAS LA UNION", "ZUNILITO", "PUEBLO NUEVO", "RIO BRAVO",
        "SAN JOSE LA MAQUINA",
    ],
    "TOTONICAPAN": [
        "TOTONICAPAN", "SAN CRISTOBAL TOTONICAPAN", "SAN FRANCISCO EL ALTO",
        "SAN ANDRES XECUL", "MOMOSTENANGO", "SANTA MARIA CHIQUIMULA",
        "SANTA LUCIA LA REFORMA", "SAN BARTOLO AGUAS CALIENTES",
    ],
    "ZACAPA": [
        "CABAÑAS", "ESTANZUELA", "GUALAN", "HUITE", "LA UNION", "RIO HONDO",
        "SAN JORGE", "SAN DIEGO", "TECULUTAN", "USUMATLAN", "ZACAPA",
    ],
}

DEPARTAMENTOS_VALIDOS = list(CATALOGO_MUNICIPIOS.keys()) + ["CIUDAD CAPITAL"]

TODOS_LOS_MUNICIPIOS = set()
for _lista in CATALOGO_MUNICIPIOS.values():
    TODOS_LOS_MUNICIPIOS.update(_lista)

# Alias verificados: nombre corto realmente usado por el MINEDUC -> nombre
# "completo" con el que aparece en algunas fuentes turísticas. Ambas formas
# son válidas y designan el mismo lugar.
ALIAS_MUNICIPIOS = {
    "GENOVA COSTA CUCA": "GENOVA",
    "LA TINTA": "SANTA CATALINA LA TINTA",
    "LANQUIN": "SAN AGUSTIN LANQUIN",
    "OLINTEPEQUE": "SAN JUAN OLINTEPEQUE",
    "SAN MIGUEL USPANTAN": "USPANTAN",
}
for _alias, _canon in ALIAS_MUNICIPIOS.items():
    TODOS_LOS_MUNICIPIOS.add(_alias)
    for _depto, _lista in CATALOGO_MUNICIPIOS.items():
        if _canon in _lista and _alias not in _lista:
            _lista.append(_alias)

print(f"Catálogo cargado: {len(DEPARTAMENTOS_VALIDOS)} departamentos, {len(TODOS_LOS_MUNICIPIOS)} municipios/alias")

Catálogo cargado: 23 departamentos, 345 municipios/alias


In [71]:
def norm_geo(x):
    """Normaliza texto para comparación (sin acentos, mayúsculas, sin espacios extra)."""
    if pd.isna(x):
        return x
    return unidecode(str(x)).upper().strip()

deptos_validos_norm = set(norm_geo(d) for d in DEPARTAMENTOS_VALIDOS)
munis_validos_norm = set(norm_geo(m) for m in TODOS_LOS_MUNICIPIOS) | {"GUATEMALA"}

deptos_en_datos = set(norm_geo(d) for d in df_clean["DEPARTAMENTO"].dropna().unique())
print("Departamentos fuera de catálogo:", deptos_en_datos - deptos_validos_norm)

Departamentos fuera de catálogo: set()


In [72]:
print("="*70)
print("VALIDACIÓN DE DOMINIO: CIUDAD CAPITAL Y ZONAS")
print("="*70)

# Para DEPARTAMENTO = 'CIUDAD CAPITAL', el campo MUNICIPIO en realidad
# contiene el número de zona (Ciudad Capital es un único municipio -Guatemala-
# subdividido en zonas administrativas, no en municipios).
es_zona = df_clean["MUNICIPIO"].astype(str).str.match(r"^ZONA\s*\d+$", na=False)
n_zonas = int(es_zona.sum())

print(f"Registros de CIUDAD CAPITAL con número de zona en MUNICIPIO: {n_zonas}")

df_clean["ZONA_CIUDAD"] = pd.NA
df_clean.loc[es_zona, "ZONA_CIUDAD"] = df_clean.loc[es_zona, "MUNICIPIO"].str.extract(r"(\d+)")[0]

df_clean["MUNICIPIO_LIMPIO"] = df_clean["MUNICIPIO"]
df_clean.loc[es_zona, "MUNICIPIO_LIMPIO"] = "GUATEMALA"

registrar(
    "MUNICIPIO",
    "Para DEPARTAMENTO = \'CIUDAD CAPITAL\' el sitio del MINEDUC reporta el número de zona en el campo MUNICIPIO en lugar del municipio real",
    "Se crea MUNICIPIO_LIMPIO = \'GUATEMALA\' (único municipio de la Ciudad Capital) y ZONA_CIUDAD conserva el número de zona original",
    n_zonas,
    "\'Ciudad Capital\' es en realidad el municipio de Guatemala subdividido en zonas administrativas, no en municipios; "
    "se preserva el detalle de zona sin perder el dato."
)

VALIDACIÓN DE DOMINIO: CIUDAD CAPITAL Y ZONAS
Registros de CIUDAD CAPITAL con número de zona en MUNICIPIO: 2161


In [73]:
print("="*70)
print("CORRECCIONES PUNTUALES DE MUNICIPIOS (verificadas manualmente)")
print("="*70)

# Cada corrección fue verificada individualmente contrastando el departamento
# de origen contra el catálogo oficial; ningún registro se elimina.
correcciones_municipio = {
    "PACHALUN": "PACHALUM",
    "SAN MIGUEL PANAM": "SAN MIGUEL PANAN",
    "SAN JOSE ACATEMPA": "ATESCATEMPA",
    "SIPACATE": "LA GOMERA",
}

justificaciones_municipio = {
    "PACHALUN": "Error tipográfico de una letra respecto al municipio oficial de Quiché \'Pachalum\'.",
    "SAN MIGUEL PANAM": "Error tipográfico respecto al municipio oficial de Suchitepéquez \'San Miguel Panán\'.",
    "SAN JOSE ACATEMPA": "\'San José Acatempa\' es una aldea del municipio de Atescatempa (Jutiapa), no un municipio; se reclasifica al municipio real.",
    "SIPACATE": "\'Sipacate\' es una aldea/balneario del municipio de La Gomera (Escuintla), no un municipio; se reclasifica al municipio real.",
}

for valor_incorrecto, valor_correcto in correcciones_municipio.items():
    mask = df_clean["MUNICIPIO_LIMPIO"] == valor_incorrecto
    n = int(mask.sum())
    if n > 0:
        df_clean.loc[mask, "MUNICIPIO_LIMPIO"] = valor_correcto
        print(f"  {valor_incorrecto!r:28s} -> {valor_correcto!r:22s} ({n} registros)")
        registrar(
            "MUNICIPIO", f"Valor fuera de dominio: \'{valor_incorrecto}\'",
            f"Reclasificado a \'{valor_correcto}\'", n,
            justificaciones_municipio[valor_incorrecto]
        )

munis_en_datos = set(norm_geo(m) for m in df_clean["MUNICIPIO_LIMPIO"].dropna().unique())
restantes = munis_en_datos - munis_validos_norm
print(f"\nMunicipios aún fuera de catálogo tras la limpieza: {len(restantes)}")
print(sorted(restantes))

CORRECCIONES PUNTUALES DE MUNICIPIOS (verificadas manualmente)
  'PACHALUN'                   -> 'PACHALUM'             (18 registros)
  'SAN MIGUEL PANAM'           -> 'SAN MIGUEL PANAN'     (1 registros)
  'SAN JOSE ACATEMPA'          -> 'ATESCATEMPA'          (12 registros)
  'SIPACATE'                   -> 'LA GOMERA'            (9 registros)

Municipios aún fuera de catálogo tras la limpieza: 0
[]


In [74]:
print("="*70)
print("CONSISTENCIA MUNICIPIO <-> DEPARTAMENTO")
print("="*70)

inconsistentes_geo = 0
for depto, grupo in df_clean.groupby("DEPARTAMENTO"):
    if depto == "CIUDAD CAPITAL":
        continue
    municipios_validos_depto = set(norm_geo(m) for m in CATALOGO_MUNICIPIOS.get(depto, []))
    municipios_depto_datos = set(norm_geo(m) for m in grupo["MUNICIPIO_LIMPIO"].dropna().unique())
    fuera = municipios_depto_datos - municipios_validos_depto
    if fuera:
        print(f"  {depto}: {fuera}")
        inconsistentes_geo += 1

print(f"\nDepartamentos con municipios inconsistentes restantes: {inconsistentes_geo}")

registrar(
    "MUNICIPIO",
    "352 valores únicos crudos vs 340 municipios oficiales: variantes no detectadas en el diagnóstico inicial",
    "Verificación cruzada contra catálogo oficial INE (22 departamentos, 340 municipios) con alias verificados para nombres cortos de uso común",
    len(correcciones_municipio),
    "Garantizar que el 100% de los registros pertenezcan a un municipio real y correspondan a su departamento."
)

CONSISTENCIA MUNICIPIO <-> DEPARTAMENTO

Departamentos con municipios inconsistentes restantes: 0


In [92]:
comparacion = pd.DataFrame({

    "Métrica":[

        "Registros",
        "Variables",
        "Valores faltantes",
        "Variables con NA",
        "Duplicados exactos",
        "Variables tipo texto",
        "Errores corregidos"

    ],

    "Antes":[

        len(df_raw),

        df_raw.shape[1],

        df_raw.isna().sum().sum(),

        (df_raw.isna().sum()>0).sum(),

        df_raw.duplicated().sum(),

        len(df_raw.select_dtypes(include=["object","string"]).columns),

        0

    ],

    "Después":[

        len(df_clean),

        df_clean.shape[1],

        df_clean.isna().sum().sum(),

        (df_clean.isna().sum()>0).sum(),

        df_clean.duplicated().sum(),

        len(df_clean.select_dtypes(include=["object","string"]).columns),

        registro["Registros afectados"].sum()

    ]

})

comparacion

,Métrica,Antes,Después
0,Registros,11891,11868
1,Variables,18,23
2,Valores faltantes,4216,14661
3,Variables con NA,17,8
4,Duplicados exactos,0,0
5,Variables tipo texto,18,21
6,Errores corregidos,0,11987


In [93]:
comparacion.to_csv(

    CLEAN_FOLDER / "comparacion_calidad.csv",

    index=False,

    encoding="utf-8-sig"

)

### 5.5 Duplicados exactos (confirmación final)

Ya se identificaron en la Parte 5 inicial; se confirma que la cifra no cambió tras
las transformaciones adicionales.

In [75]:
print("="*70)
print("5.5 DUPLICADOS EXACTOS (confirmación)")
print("="*70)

dup_exactos_final = df_clean.duplicated().sum()
print(f"Duplicados exactos: {dup_exactos_final}")

5.5 DUPLICADOS EXACTOS (confirmación)
Duplicados exactos: 0


### 5.6 Duplicados parciales (similitud de cadenas)

Se utiliza **RapidFuzz** (`token_sort_ratio`) para comparar el nombre del establecimiento
dentro de cada grupo Departamento+Municipio, exigiendo además que la dirección sea similar
o que el teléfono coincida, antes de considerar un par como candidato a duplicado. **No se
elimina ningún registro**: se documenta y se deja marcado para revisión manual, tal como
indica la guía.

In [76]:
print("="*70)
print("5.6 DUPLICADOS PARCIALES (RapidFuzz)")
print("="*70)

def norm_txt(x):
    if pd.isna(x):
        return ""
    x = unidecode(str(x)).upper().strip()
    return " ".join(x.split())

df_clean["_EST_NORM"] = df_clean["ESTABLECIMIENTO"].apply(norm_txt)
df_clean["_DIR_NORM"] = df_clean["DIRECCION"].apply(norm_txt)
df_clean["_TEL_NORM"] = df_clean["TELEFONO_NORMALIZADO"].apply(norm_txt)

candidatos = []
for (depto, muni), grupo in df_clean.groupby(["DEPARTAMENTO", "MUNICIPIO_LIMPIO"]):
    idxs = grupo.index.tolist()
    n = len(idxs)
    if n < 2:
        continue
    for i in range(n):
        for j in range(i + 1, n):
            a, b = idxs[i], idxs[j]
            if not df_clean.loc[a, "_EST_NORM"] or not df_clean.loc[b, "_EST_NORM"]:
                continue
            score_est = fuzz.token_sort_ratio(df_clean.loc[a, "_EST_NORM"], df_clean.loc[b, "_EST_NORM"])
            if score_est < 92:
                continue
            score_dir = (
                fuzz.token_sort_ratio(df_clean.loc[a, "_DIR_NORM"], df_clean.loc[b, "_DIR_NORM"])
                if df_clean.loc[a, "_DIR_NORM"] and df_clean.loc[b, "_DIR_NORM"] else 0
            )
            mismo_tel = df_clean.loc[a, "_TEL_NORM"] == df_clean.loc[b, "_TEL_NORM"] and df_clean.loc[a, "_TEL_NORM"] != ""
            if score_dir >= 70 or mismo_tel:
                candidatos.append({
                    "codigo_a": df_clean.loc[a, "CODIGO"], "codigo_b": df_clean.loc[b, "CODIGO"],
                    "establecimiento_a": df_clean.loc[a, "ESTABLECIMIENTO"],
                    "establecimiento_b": df_clean.loc[b, "ESTABLECIMIENTO"],
                    "score_nombre": round(score_est, 1), "score_direccion": round(score_dir, 1),
                    "mismo_telefono": mismo_tel,
                    "misma_jornada": df_clean.loc[a, "JORNADA"] == df_clean.loc[b, "JORNADA"],
                    "mismo_plan": df_clean.loc[a, "PLAN"] == df_clean.loc[b, "PLAN"],
                    "idx_a": a, "idx_b": b,
                })

candidatos_df = pd.DataFrame(candidatos)
print(f"Candidatos totales (nombre >=92 y dirección/teléfono coincidente): {len(candidatos_df)}")

# Se consideran "posibles duplicados" únicamente los que además comparten
# JORNADA y PLAN (mayor probabilidad de ser el mismo registro repetido, y no
# la misma institución ofreciendo un programa distinto).
posibles_duplicados_df = candidatos_df[candidatos_df["misma_jornada"] & candidatos_df["mismo_plan"]].copy()
print(f"Candidatos con misma JORNADA y PLAN (posibles duplicados): {len(posibles_duplicados_df)}")

idx_posibles_duplicados = set(posibles_duplicados_df["idx_a"]) | set(posibles_duplicados_df["idx_b"])
df_clean["POSIBLE_DUPLICADO"] = df_clean.index.isin(idx_posibles_duplicados)
print(f"Registros marcados como POSIBLE_DUPLICADO: {df_clean['POSIBLE_DUPLICADO'].sum()}")

registrar(
    "DATASET",
    "Duplicados parciales: mismo establecimiento con variaciones de escritura o registrado más de una vez",
    "Detección con RapidFuzz (token_sort_ratio) agrupando por DEPARTAMENTO+MUNICIPIO; se marca POSIBLE_DUPLICADO=True "
    "cuando nombre >=92%, coincide dirección (>=70%) o teléfono, Y coincide JORNADA y PLAN",
    int(df_clean["POSIBLE_DUPLICADO"].sum()),
    "No se elimina ningún registro automáticamente: muchos pares corresponden a la misma institución que ofrece más de un "
    "programa u horario bajo códigos distintos, lo cual es válido. Se deja marcado para revisión manual, conforme lo "
    "solicita la guía."
)

print("\nMuestra de pares candidatos (primeros 10):")
posibles_duplicados_df[["codigo_a", "establecimiento_a", "codigo_b", "establecimiento_b", "score_nombre", "score_direccion"]].head(10)

5.6 DUPLICADOS PARCIALES (RapidFuzz)
Candidatos totales (nombre >=92 y dirección/teléfono coincidente): 9164
Candidatos con misma JORNADA y PLAN (posibles duplicados): 1560
Registros marcados como POSIBLE_DUPLICADO: 2525

Muestra de pares candidatos (primeros 10):


,codigo_a,establecimiento_a,codigo_b,establecimiento_b,score_nombre,score_direccion
0,16-14-0113-46,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,16-14-0114-46,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,100.0,75.9
1,16-14-0115-46,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,16-14-0116-46,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,100.0,100.0
3,16-13-0225-46,LICEO MIXTO EN COMPUTACION DEL NORTE,16-13-0226-46,LICEO MIXTO EN COMPUTACION DEL NORTE,100.0,100.0
22,16-01-0481-46,LICEO AMERICANO DEL NORTE,16-01-0664-46,LICEO AMERICANO DEL NORTE,100.0,82.6
24,16-01-0559-46,INSTITUTO NORMAL PRIMARIA BILINGUE INTERCULTUR...,16-01-8849-46,INSTITUTO NORMAL PRIMARIA BILINGUE INTERCULTUR...,100.0,100.0
33,16-01-0587-46,LICEO MIXTO EN COMPUTACION DEL NORTE,16-01-0588-46,LICEO MIXTO EN COMPUTACION DEL NORTE,100.0,100.0
48,16-01-0666-46,"COLEGIO DE INFORMÁTICA ""CENINFAV""",16-01-9851-46,COLEGIO DE INFORMATICA CENINFAV,96.9,71.8
81,16-01-0926-46,INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFICADA,16-01-0927-46,INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFICADA,100.0,87.5
85,16-01-1046-46,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,16-01-1047-46,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,100.0,71.4
129,16-16-0135-46,LICEO CRISTIANO EBENEZER,16-16-0139-46,LICEO CRISTIANO EBENEZER,100.0,100.0


In [77]:
# Se guarda la lista completa de candidatos para revisión manual posterior
posibles_duplicados_df.drop(columns=["idx_a", "idx_b"]).to_csv(
    CLEAN_FOLDER / "posibles_duplicados.csv",
    index=False,
    encoding="utf-8-sig"
)
print("Lista completa de posibles duplicados guardada para revisión manual.")

Lista completa de posibles duplicados guardada para revisión manual.


### 5.7 Consistencia entre variables

In [78]:
print("="*70)
print("5.7 CONSISTENCIA ENTRE VARIABLES")
print("="*70)

abiertas_sin_jornada = ((df_clean["STATUS"] == "ABIERTA") & (df_clean["JORNADA"] == "SIN JORNADA")).sum()
print(f"Establecimientos ABIERTOS sin jornada definida: {abiertas_sin_jornada}")
print("(se documenta, no se corrige: es una limitación de la fuente, no hay forma confiable de inferir la jornada real)")

registrar(
    "STATUS / JORNADA",
    "Establecimientos con STATUS=\'ABIERTA\' pero JORNADA=\'SIN JORNADA\'",
    "Sin corrección automática; se documenta como limitación de la fuente",
    int(abiertas_sin_jornada),
    "No existe una regla confiable para inferir la jornada real sin acceder a información adicional del MINEDUC."
)

# Limpieza de columnas auxiliares que ya cumplieron su propósito
df_clean = df_clean.drop(columns=["_EST_NORM", "_DIR_NORM", "_TEL_NORM"])
print("\nColumnas auxiliares de normalización eliminadas. Columnas actuales:")
print(df_clean.columns.tolist())

5.7 CONSISTENCIA ENTRE VARIABLES
Establecimientos ABIERTOS sin jornada definida: 1037
(se documenta, no se corrige: es una limitación de la fuente, no hay forma confiable de inferir la jornada real)

Columnas auxiliares de normalización eliminadas. Columnas actuales:
['CODIGO', 'DISTRITO', 'DEPARTAMENTO', 'MUNICIPIO', 'ESTABLECIMIENTO', 'DIRECCION', 'TELEFONO', 'SUPERVISOR', 'DIRECTOR', 'NIVEL', 'SECTOR', 'AREA', 'STATUS', 'MODALIDAD', 'JORNADA', 'PLAN', 'DEPARTAMENTAL', 'ARCHIVO_ORIGEN', 'TELEFONO_NORMALIZADO', 'TELEFONO_VALIDO', 'ZONA_CIUDAD', 'MUNICIPIO_LIMPIO', 'POSIBLE_DUPLICADO']


# Parte 6  - Registro de Transformaciones completo

Se vuelve a generar el registro de transformaciones incluyendo todas las operaciones
de la Parte 5 completa (limpieza inicial + limpieza avanzada), sobrescribiendo el
archivo intermedio guardado anteriormente.

In [79]:
registro_final = pd.DataFrame(registro_transformaciones)
registro_final.index = registro_final.index + 1

registro_final.to_csv(
    CLEAN_FOLDER / "registro_transformaciones.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Registro de transformaciones actualizado: {len(registro_final)} operaciones documentadas.")
registro_final

Registro de transformaciones actualizado: 51 operaciones documentadas.


,Variable,Problema,Transformación,Registros afectados,Justificación
1,TODAS LAS COLUMNAS,Nombres inconsistentes,Normalización de nombres,18,Facilitar el acceso mediante pandas.
2,CODIGO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
3,DISTRITO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
4,DEPARTAMENTO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
5,MUNICIPIO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
6,ESTABLECIMIENTO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
7,DIRECCION,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
8,TELEFONO,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
9,SUPERVISOR,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.
10,DIRECTOR,"Espacios iniciales, finales o múltiples",Eliminación de espacios innecesarios,0,Uniformar el formato sin alterar el contenido.


# Parte 7 - Validación del Conjunto Limpio

Se ejecutan pruebas automáticas que verifican que el conjunto de datos cumple las
reglas de calidad establecidas al inicio del proyecto.

In [80]:
resultados_validacion = []

def validar(nombre, condicion, detalle=""):
    estado = "OK" if condicion else "FALLA"
    resultados_validacion.append({"Prueba": nombre, "Resultado": estado, "Detalle": detalle})
    print(f"[{estado}] {nombre} {detalle}")

print("="*70)
print("PRUEBAS DE VALIDACIÓN")
print("="*70)

# 1. Sin duplicados exactos
validar("Sin registros duplicados exactos", df_clean.duplicated().sum() == 0,
         f"(duplicados: {df_clean.duplicated().sum()})")

# 2. Sin espacios al inicio/final en texto
espacios_mal = 0
for c in df_clean.select_dtypes(include=["string"]).columns:
    espacios_mal += (df_clean[c].dropna().astype(str) != df_clean[c].dropna().astype(str).str.strip()).sum()
validar("Sin espacios al inicio/final en variables de texto", espacios_mal == 0,
         f"(afectados: {espacios_mal})")

# 3. Teléfonos con formato consistente (8 dígitos donde aplique)
tel_malos = df_clean.loc[df_clean["TELEFONO_VALIDO"], "TELEFONO_NORMALIZADO"].apply(
    lambda x: any(len(p) != 8 or not p.isdigit() for p in str(x).split("; "))
).sum()
validar("Teléfonos válidos tienen formato de 8 dígitos", tel_malos == 0,
         f"(incorrectos: {tel_malos})")

# 4. Departamentos y municipios pertenecen al catálogo oficial
deptos_ok = set(norm_geo(d) for d in df_clean["DEPARTAMENTO"].dropna().unique()) <= deptos_validos_norm
munis_ok = set(norm_geo(m) for m in df_clean["MUNICIPIO_LIMPIO"].dropna().unique()) <= munis_validos_norm
validar("Departamentos pertenecen al catálogo oficial", deptos_ok)
validar("Municipios (limpios) pertenecen al catálogo oficial", munis_ok)

# 5. Variables derivadas con tipo correcto
tipos_ok = (df_clean["TELEFONO_VALIDO"].dtype == bool) and (df_clean["POSIBLE_DUPLICADO"].dtype == bool)
validar("Variables booleanas derivadas con tipo correcto", tipos_ok)

# 6. Sin categorías duplicadas por diferencias de escritura
categoricas_check = ["SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN"]
sin_dup_categoria = True
for c in categoricas_check:
    valores = df_clean[c].dropna().unique()
    normalizados = set(norm_geo(v) for v in valores)
    if len(normalizados) != len(valores):
        sin_dup_categoria = False
validar("Sin categorías duplicadas por escritura en variables clave", sin_dup_categoria)

# 7. Sin valores inválidos detectados en el diagnóstico inicial
validar("Sin valores de MUNICIPIO fuera de dominio", len(restantes) == 0)
validar("Sin filas completamente vacías", df_clean[cols_contenido].isna().all(axis=1).sum() == 0)

reporte_validacion = pd.DataFrame(resultados_validacion)
print("\nResumen de validación:")
reporte_validacion

PRUEBAS DE VALIDACIÓN
[OK] Sin registros duplicados exactos (duplicados: 0)
[OK] Sin espacios al inicio/final en variables de texto (afectados: 0)
[OK] Teléfonos válidos tienen formato de 8 dígitos (incorrectos: 0)
[OK] Departamentos pertenecen al catálogo oficial 
[OK] Municipios (limpios) pertenecen al catálogo oficial 
[OK] Variables booleanas derivadas con tipo correcto 
[OK] Sin categorías duplicadas por escritura en variables clave 
[OK] Sin valores de MUNICIPIO fuera de dominio 
[OK] Sin filas completamente vacías 

Resumen de validación:


,Prueba,Resultado,Detalle
0,Sin registros duplicados exactos,OK,(duplicados: 0)
1,Sin espacios al inicio/final en variables de t...,OK,(afectados: 0)
2,Teléfonos válidos tienen formato de 8 dígitos,OK,(incorrectos: 0)
3,Departamentos pertenecen al catálogo oficial,OK,
4,Municipios (limpios) pertenecen al catálogo of...,OK,
5,Variables booleanas derivadas con tipo correcto,OK,
6,Sin categorías duplicadas por escritura en var...,OK,
7,Sin valores de MUNICIPIO fuera de dominio,OK,
8,Sin filas completamente vacías,OK,


In [81]:
todas_ok = (reporte_validacion["Resultado"] == "OK").all()
print(f"¿Todas las pruebas de calidad pasaron? {todas_ok}")

reporte_validacion.to_csv(CLEAN_FOLDER / "reporte_validacion.csv", index=False, encoding="utf-8-sig")
print("Reporte de validación guardado.")

¿Todas las pruebas de calidad pasaron? True
Reporte de validación guardado.


# Parte 8 - Informe de Calidad de los Datos

Se compara el estado del conjunto de datos antes y después de la limpieza.

In [82]:
cols_originales = df_raw.columns.tolist()

faltantes_antes = int(df_raw.isna().sum().sum())
faltantes_despues_comparable = int(df_clean[cols_originales].isna().sum().sum())

cat_inconsist_antes = 352 - 340   # municipios crudos únicos vs. 340 municipios oficiales
cat_inconsist_despues = 0

informe_calidad = pd.DataFrame({
    "Métrica": [
        "Registros", "Variables", "Valores faltantes (18 var. originales)", "Variables con NA",
        "Duplicados exactos", "Posibles duplicados", "Variables con formato inconsistente",
        "Variables con tipo incorrecto", "Categorías inconsistentes", "Errores corregidos"
    ],
    "Antes": [
        len(df_raw), df_raw.shape[1], faltantes_antes, int((df_raw.isna().sum() > 0).sum()),
        int(df_raw.duplicated().sum()), "No evaluado (sin normalizar)",
        3,   # ESTABLECIMIENTO/DIRECCION (espacios y mayúsculas), TELEFONO (formatos), MUNICIPIO (Ciudad Capital)
        1,   # TELEFONO tratado de forma no consistente en la fuente
        cat_inconsist_antes,
        0
    ],
    "Después": [
        len(df_clean), df_clean.shape[1], faltantes_despues_comparable, int((df_clean[cols_originales].isna().sum() > 0).sum()),
        int(df_clean.duplicated().sum()), int(df_clean["POSIBLE_DUPLICADO"].sum()),
        0,
        0,
        cat_inconsist_despues,
        len(registro_transformaciones)
    ]
})

print(informe_calidad.to_string(index=False))
informe_calidad

                               Métrica                        Antes  Después
                             Registros                        11891    11868
                             Variables                           18       23
Valores faltantes (18 var. originales)                         4216     3903
                      Variables con NA                           17        6
                    Duplicados exactos                            0        0
                   Posibles duplicados No evaluado (sin normalizar)     2525
   Variables con formato inconsistente                            3        0
         Variables con tipo incorrecto                            1        0
             Categorías inconsistentes                           12        0
                    Errores corregidos                            0       51


,Métrica,Antes,Después
0,Registros,11891,11868
1,Variables,18,23
2,Valores faltantes (18 var. originales),4216,3903
3,Variables con NA,17,6
4,Duplicados exactos,0,0
5,Posibles duplicados,No evaluado (sin normalizar),2525
6,Variables con formato inconsistente,3,0
7,Variables con tipo incorrecto,1,0
8,Categorías inconsistentes,12,0
9,Errores corregidos,0,51


In [83]:
print(
    "Nota: la comparación de \'valores faltantes\' usa únicamente las 18 variables originales "
    "para que la comparación sea justa. El dataset final agrega 5 columnas derivadas "
    "(TELEFONO_NORMALIZADO/TELEFONO, TELEFONO_VALIDO, ZONA_CIUDAD, MUNICIPIO_LIMPIO/MUNICIPIO, "
    "POSIBLE_DUPLICADO) documentadas en el Libro de Códigos; ZONA_CIUDAD por diseño solo aplica "
    "a los registros de Ciudad Capital."
)

informe_calidad.to_csv(CLEAN_FOLDER / "informe_calidad.csv", index=False, encoding="utf-8-sig")
print("Informe de calidad guardado.")

Nota: la comparación de 'valores faltantes' usa únicamente las 18 variables originales para que la comparación sea justa. El dataset final agrega 5 columnas derivadas (TELEFONO_NORMALIZADO/TELEFONO, TELEFONO_VALIDO, ZONA_CIUDAD, MUNICIPIO_LIMPIO/MUNICIPIO, POSIBLE_DUPLICADO) documentadas en el Libro de Códigos; ZONA_CIUDAD por diseño solo aplica a los registros de Ciudad Capital.
Informe de calidad guardado.


# Parte 9 - Generación del Conjunto Limpio Final

Se genera el conjunto de datos final, con estructura consistente, tipos de datos
correctos, nombres de variables descriptivos y formato uniforme.

In [84]:
df_final = df_clean.rename(columns={
    "MUNICIPIO": "MUNICIPIO_ORIGINAL",
    "MUNICIPIO_LIMPIO": "MUNICIPIO",
    "TELEFONO": "TELEFONO_ORIGINAL",
    "TELEFONO_NORMALIZADO": "TELEFONO",
})

columnas_finales = [
    "CODIGO", "DISTRITO", "DEPARTAMENTO", "MUNICIPIO", "MUNICIPIO_ORIGINAL", "ZONA_CIUDAD",
    "ESTABLECIMIENTO", "DIRECCION", "TELEFONO", "TELEFONO_ORIGINAL", "TELEFONO_VALIDO",
    "SUPERVISOR", "DIRECTOR", "NIVEL", "SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA",
    "PLAN", "DEPARTAMENTAL", "POSIBLE_DUPLICADO", "ARCHIVO_ORIGEN",
]
df_final = df_final[columnas_finales]

print(f"Dimensiones del conjunto final: {df_final.shape}")
df_final.dtypes

Dimensiones del conjunto final: (11868, 23)


0
CODIGO                string
DISTRITO              string
DEPARTAMENTO          string
MUNICIPIO             string
MUNICIPIO_ORIGINAL    string
ZONA_CIUDAD           object
ESTABLECIMIENTO       string
DIRECCION             string
TELEFONO                 str
TELEFONO_ORIGINAL     string
TELEFONO_VALIDO         bool
SUPERVISOR            string
DIRECTOR              string
NIVEL                 string
SECTOR                string
AREA                  string
STATUS                string
MODALIDAD             string
JORNADA               string
PLAN                  string
DEPARTAMENTAL         string
POSIBLE_DUPLICADO       bool
ARCHIVO_ORIGEN        string
dtype: object

In [85]:
csv_final_path = CLEAN_FOLDER / "establecimientos_diversificado_limpio.csv"

df_final.to_csv(
    csv_final_path,
    index=False,
    encoding="utf-8-sig"
)

print("Conjunto de datos limpio generado correctamente.")
print(csv_final_path)
df_final.head()

Conjunto de datos limpio generado correctamente.
data\clean\establecimientos_diversificado_limpio.csv


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,MUNICIPIO_ORIGINAL,ZONA_CIUDAD,ESTABLECIMIENTO,DIRECCION,TELEFONO,TELEFONO_ORIGINAL,...,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL,POSIBLE_DUPLICADO,ARCHIVO_ORIGEN
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,COBAN,<NA>,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,NaN,<NA>,...,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ,False,Alta_Verapaz
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COBAN,<NA>,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,77945104,...,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,False,Alta_Verapaz
2,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COBAN,<NA>,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,58015763,58015763,...,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,False,Alta_Verapaz
3,16-01-0140-46,16-01-0926,ALTA VERAPAZ,COBAN,COBAN,<NA>,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,78232301,78232301,...,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,False,Alta_Verapaz
4,16-01-0141-46,16-01-0924,ALTA VERAPAZ,COBAN,COBAN,<NA>,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,40389968,40389968,...,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,False,Alta_Verapaz


# Parte 10 - Libro de Códigos

Documenta, para cada variable del conjunto final: descripción, tipo de dato, dominio
permitido, tratamiento aplicado durante la limpieza, variables derivadas, fecha de
extracción, fuente y versión del conjunto.

In [86]:
libro_codigos = [
    dict(Variable="CODIGO", Descripcion="Código único del establecimiento asignado por el MINEDUC (formato depto-distrito-correlativo-nivel)",
         Tipo="string", Dominio="Patrón NN-NN-NNNN-NN", Tratamiento="Sin cambios (ya consistente)", Derivada="No"),
    dict(Variable="DISTRITO", Descripcion="Código de distrito escolar",
         Tipo="string", Dominio="Texto libre (formato variable)", Tratamiento="Trim de espacios", Derivada="No"),
    dict(Variable="DEPARTAMENTO", Descripcion="Departamento donde opera el establecimiento (clasificación del MINEDUC)",
         Tipo="string", Dominio="22 departamentos + \'CIUDAD CAPITAL\'", Tratamiento="Trim, mayúsculas, validado contra catálogo INE", Derivada="No"),
    dict(Variable="MUNICIPIO", Descripcion="Municipio real del establecimiento, ya corregido",
         Tipo="string", Dominio="340 municipios oficiales de Guatemala", Tratamiento="CIUDAD CAPITAL->GUATEMALA; typos y aldeas reclasificadas a su municipio real", Derivada="Sí (a partir de MUNICIPIO_ORIGINAL)"),
    dict(Variable="MUNICIPIO_ORIGINAL", Descripcion="Valor de MUNICIPIO tal como venía en la fuente, antes de corrección",
         Tipo="string", Dominio="Texto libre", Tratamiento="Se conserva para trazabilidad", Derivada="No"),
    dict(Variable="ZONA_CIUDAD", Descripcion="Número de zona de la Ciudad de Guatemala, cuando aplica",
         Tipo="string (numérico)", Dominio="1-25 o nulo", Tratamiento="Extraída de MUNICIPIO_ORIGINAL para registros de Ciudad Capital", Derivada="Sí"),
    dict(Variable="ESTABLECIMIENTO", Descripcion="Nombre del centro educativo",
         Tipo="string", Dominio="Texto libre", Tratamiento="Trim, mayúsculas uniformes", Derivada="No"),
    dict(Variable="DIRECCION", Descripcion="Dirección física del establecimiento",
         Tipo="string", Dominio="Texto libre", Tratamiento="Trim, mayúsculas uniformes", Derivada="No"),
    dict(Variable="TELEFONO", Descripcion="Teléfono(s) normalizados de 8 dígitos, separados por \'; \'",
         Tipo="string", Dominio="8 dígitos numéricos (uno o más)", Tratamiento="Extraído por regex desde TELEFONO_ORIGINAL", Derivada="Sí (a partir de TELEFONO_ORIGINAL)"),
    dict(Variable="TELEFONO_ORIGINAL", Descripcion="Teléfono tal como venía en la fuente (texto libre, múltiples números, etc.)",
         Tipo="string", Dominio="Texto libre", Tratamiento="Se conserva para trazabilidad", Derivada="No"),
    dict(Variable="TELEFONO_VALIDO", Descripcion="Indica si se encontró al menos un teléfono de 8 dígitos válido",
         Tipo="boolean", Dominio="True / False", Tratamiento="—", Derivada="Sí"),
    dict(Variable="SUPERVISOR", Descripcion="Nombre del supervisor educativo responsable",
         Tipo="string", Dominio="Texto libre", Tratamiento="Trim, mayúsculas uniformes", Derivada="No"),
    dict(Variable="DIRECTOR", Descripcion="Nombre del director del establecimiento",
         Tipo="string", Dominio="Texto libre", Tratamiento="Trim, mayúsculas uniformes", Derivada="No"),
    dict(Variable="NIVEL", Descripcion="Nivel educativo (filtro de obtención)",
         Tipo="string", Dominio="\'DIVERSIFICADO\' (único valor)", Tratamiento="Sin cambios; validado como constante", Derivada="No"),
    dict(Variable="SECTOR", Descripcion="Sector administrativo del establecimiento",
         Tipo="string", Dominio="PRIVADO, OFICIAL, COOPERATIVA, MUNICIPAL", Tratamiento="Trim, mayúsculas", Derivada="No"),
    dict(Variable="AREA", Descripcion="Área geográfica del establecimiento",
         Tipo="string", Dominio="URBANA, RURAL, SIN ESPECIFICAR", Tratamiento="Trim, mayúsculas", Derivada="No"),
    dict(Variable="STATUS", Descripcion="Estado operativo del establecimiento",
         Tipo="string", Dominio="ABIERTA, CERRADA TEMPORALMENTE, CERRADA DEFINITIVAMENTE, TEMPORAL TITULOS, TEMPORAL NOMBRAMIENTO",
         Tratamiento="Trim, mayúsculas", Derivada="No"),
    dict(Variable="MODALIDAD", Descripcion="Modalidad lingüística del establecimiento",
         Tipo="string", Dominio="MONOLINGUE, BILINGUE", Tratamiento="Trim, mayúsculas", Derivada="No"),
    dict(Variable="JORNADA", Descripcion="Jornada en la que opera el establecimiento",
         Tipo="string", Dominio="MATUTINA, VESPERTINA, NOCTURNA, DOBLE, INTERMEDIA, SIN JORNADA", Tratamiento="Trim, mayúsculas", Derivada="No"),
    dict(Variable="PLAN", Descripcion="Plan de estudios del programa",
         Tipo="string", Dominio="DIARIO(REGULAR), FIN DE SEMANA, SABATINO, DOMINICAL, SEMIPRESENCIAL (y variantes), A DISTANCIA, VIRTUAL A DISTANCIA, MIXTO, IRREGULAR, INTERCALADO",
         Tratamiento="Trim, mayúsculas", Derivada="No"),
    dict(Variable="DEPARTAMENTAL", Descripcion="Dirección Departamental de Educación que supervisa el establecimiento (división administrativa del MINEDUC, distinta de DEPARTAMENTO)",
         Tipo="string", Dominio="26 direcciones departamentales (p. ej. GUATEMALA NORTE/SUR/ORIENTE/OCCIDENTE)", Tratamiento="Trim, mayúsculas", Derivada="No"),
    dict(Variable="POSIBLE_DUPLICADO", Descripcion="Indica si el registro fue detectado como posible duplicado parcial de otro",
         Tipo="boolean", Dominio="True / False", Tratamiento="—", Derivada="Sí (RapidFuzz sobre ESTABLECIMIENTO/DIRECCION/TELEFONO); ningún registro fue eliminado"),
    dict(Variable="ARCHIVO_ORIGEN", Descripcion="Nombre del archivo Excel de origen (identifica el departamento descargado)",
         Tipo="string", Dominio="23 valores (uno por departamento descargado)", Tratamiento="Sin cambios", Derivada="No"),
]

libro_df = pd.DataFrame(libro_codigos)
libro_df["Fecha_extraccion"] = pd.Timestamp.today().strftime("%Y-%m-%d")
libro_df["Fuente"] = "MINEDUC Guatemala - Buscador de Establecimientos Educativos (http://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/)"
libro_df["Version_dataset"] = "1.0"

libro_df

,Variable,Descripcion,Tipo,Dominio,Tratamiento,Derivada,Fecha_extraccion,Fuente,Version_dataset
0,CODIGO,Código único del establecimiento asignado por ...,string,Patrón NN-NN-NNNN-NN,Sin cambios (ya consistente),No,2026-07-23,MINEDUC Guatemala - Buscador de Establecimient...,1.0
1,DISTRITO,Código de distrito escolar,string,Texto libre (formato variable),Trim de espacios,No,2026-07-23,MINEDUC Guatemala - Buscador de Establecimient...,1.0
2,DEPARTAMENTO,Departamento donde opera el establecimiento (c...,string,22 departamentos + 'CIUDAD CAPITAL',"Trim, mayúsculas, validado contra catálogo INE",No,2026-07-23,MINEDUC Guatemala - Buscador de Establecimient...,1.0
3,MUNICIPIO,"Municipio real del establecimiento, ya corregido",string,340 municipios oficiales de Guatemala,CIUDAD CAPITAL->GUATEMALA; typos y aldeas recl...,Sí (a partir de MUNICIPIO_ORIGINAL),2026-07-23,MINEDUC Guatemala - Buscador de Establecimient...,1.0
4,MUNICIPIO_ORIGINAL,Valor de MUNICIPIO tal como venía en la fuente...,string,Texto libre,Se conserva para trazabilidad,No,2026-07-23,MINEDUC Guatemala - Buscador de Establecimient...,1.0
5,ZONA_CIUDAD,"Número de zona de la Ciudad de Guatemala, cuan...",string (numérico),1-25 o nulo,Extraída de MUNICIPIO_ORIGINAL para registros ...,Sí,2026-07-23,MINEDUC Guatemala - Buscador de Establecimient...,1.0
6,ESTABLECIMIENTO,Nombre del centro educativo,string,Texto libre,"Trim, mayúsculas uniformes",No,2026-07-23,MINEDUC Guatemala - Buscador de Establecimient...,1.0
7,DIRECCION,Dirección física del establecimiento,string,Texto libre,"Trim, mayúsculas uniformes",No,2026-07-23,MINEDUC Guatemala - Buscador de Establecimient...,1.0
8,TELEFONO,"Teléfono(s) normalizados de 8 dígitos, separad...",string,8 dígitos numéricos (uno o más),Extraído por regex desde TELEFONO_ORIGINAL,Sí (a partir de TELEFONO_ORIGINAL),2026-07-23,MINEDUC Guatemala - Buscador de Establecimient...,1.0
9,TELEFONO_ORIGINAL,Teléfono tal como venía en la fuente (texto li...,string,Texto libre,Se conserva para trazabilidad,No,2026-07-23,MINEDUC Guatemala - Buscador de Establecimient...,1.0


In [87]:
libro_path = CLEAN_FOLDER / "libro_codigos.csv"

libro_df.to_csv(
    libro_path,
    index=False,
    encoding="utf-8-sig"
)

print("Libro de códigos guardado correctamente.")
print(libro_path)

Libro de códigos guardado correctamente.
data\clean\libro_codigos.csv
